In [5]:
"""大白话：这个代码格是“说明书 + 入口”。

你给的口径来源：`BOSS衍生.xlsx`（Sheet1），本 Notebook 会用 `衍生原始数据_331条.csv` 作为原始数据，严格按 Excel 里每一条“中文释义 + 逻辑描述”去计算特征。

【总体产出】
- 产出一张特征表 `features_df`：每个 `apply_id` 一行，列是 BOSS 板块的衍生特征。
- **特征命名**：所有特征列名统一加前后缀：`cdc_{原特征名}_607`。
- 默认不写任何文件（不生成 CSV / 不生成新的 notebook / 不落 outputs）。

【特征量级】
- Excel 中一共有 N 条特征（不含表头）。本 Notebook 会在运行后打印 `features_df.shape`，确保列数与 Excel 条目对齐。

【分了几个小版块（按数据来源分块）】
1) consultas（查询板块）：
   - 字段：`fechaConsulta`（查询日期）、`importeCredito`（合同金额）、`tipoCredito`（查询的信贷类型）。
   - 典型特征：
     - 近 30/60/90/180/360/720 天窗口内：查询金额 sum / max / mean；
     - 全量：车贷查询次数、信用卡查询次数、查询类型数量、平均查询间隔（天）、查询频率（次/月）、查询激增标记等。

2) creditos（信贷板块）：
   - 字段：`fechaReporte`（上报日期）、`fechaAperturaCuenta`（开户）、`fechaCierreCuenta`（关闭）、`saldoActual`、`limiteCredito`、`saldoVencido`、
           `tipoCuenta`、`tipoCredito`、`tipoResponsabilidad`、`frecuenciaPagos`、以及多个“最近一次事件日期”（如 fechaUltimoPago/fechaUltimaCompra/fechaActualizacion/fechaPeorAtraso/ultimaFechaSaldoCero）。
   - 典型特征：
     - 账户数/状态类：总账户数、开放/关闭/零余额、个人/联名/担保等责任类型计数与余额汇总；
     - 使用率类：账户平均使用率、信用卡使用率、高使用率账户数、总使用率（分子/分母汇总）；
     - 收入比类：授信收入比、负债收入比、逾期收入比、还款收入比；
     - 时效类：距最后销户/查询/逾期/还款/消费/报告/更新/零余额天数；
     - 其它：最差逾期状态、还款期数均值/总和、已报告还款次数均值/总和、还款频率众数映射等。

3) empleos（工作信息）：
   - 字段：`salarioMensual`、`fechaContratacion`、`fechaUltimoDiaEmpleo`、`fechaVerificacionEmpleo`、`nombreEmpresa`。
   - 典型特征：工作记录数、平均/最长/当前工作月数、平均/当前/历史最高月薪、工作变更次数、工作稳定性得分、薪资已验证标记等。

4) domicilios（住址信息）：
   - 字段：`fechaResidencia`、`estado`、`direccion`。
   - 典型特征：平均/最长/当前居住月数、州数量、近期搬迁标记、居住稳定性得分等。

【窗口口径（你口头确认）】
- 窗口终点 = “截止日期”。在原始数据里通常就是 `request_time`（每个样本自己的截止时间）。
- 窗口长度：30 / 60 / 90 / 180 / 360 / 720（天），窗口范围是：
  - 截止日期往前推 N 天：`cutoff - N天 <= 日期字段 <= cutoff`
- 信贷板块（creditos）所有“窗口内”特征：统一用 `fechaReporte` 作为窗口筛选日期字段。
- 查询板块（consultas）所有“窗口内”特征：统一用 `fechaConsulta` 作为窗口筛选日期字段。

【时间差换算】
- 统一按：`(cutoff - date).total_seconds() / 86400` 换算为“天”（可带小数）。
- 月数统一按：`days / 30` 换算为“月”（与 Excel 描述保持一致）。

接下来：读取 331 条原始数据 + 读取 Excel（仅用于核对条目数量/口径），并解析 response_body。
"""

# 大白话：执行下一行代码：from __future__ import annotations
from __future__ import annotations

# 大白话：执行下一行代码：import json
import json
# 大白话：执行下一行代码：from pathlib import Path
from pathlib import Path

# 大白话：执行下一行代码：import numpy as np
import numpy as np
# 大白话：执行下一行代码：import pandas as pd
import pandas as pd

# =========================
# 1) 路径与读取
# =========================
# 大白话：执行下一行代码：BASE_DIR = Path(\".\")
BASE_DIR = Path(".")

# 大白话：为了让所有板块口径一致，这里优先读取你筛选后的 pickle：cdc_pickle_pass_fpd7.pkl。
# 大白话：如果你想回到 331 条 CSV，只需要把 USE_PICKLE 改成 False。
USE_PICKLE = True  # True=优先读 pickle；False=读 CSV
PICKLE_PATH = BASE_DIR / "cdc_pickle_pass_fpd7.pkl"

# 大白话：兜底 CSV（只有在 pickle 不存在或 USE_PICKLE=False 时才用）。
# 大白话：执行下一行代码：CSV_PATH = BASE_DIR / \"衍生原始数据_331条.csv\"
CSV_PATH = BASE_DIR / "衍生原始数据_331条.csv"
# 大白话：执行下一行代码：XLSX_PATH = BASE_DIR / \"BOSS衍生.xlsx\"
XLSX_PATH = BASE_DIR / "BOSS衍生.xlsx"

# 大白话：读入数据（优先 pickle；否则读 CSV）。
if USE_PICKLE and PICKLE_PATH.exists():
    raw_df = pd.read_pickle(PICKLE_PATH)
    print("[INFO] loaded pickle:", PICKLE_PATH)
else:
    raw_df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    print("[INFO] loaded csv:", CSV_PATH)

# 大白话：cdc_pickle_pass_fpd7.pkl 里一般用 apply_time 表示“截止日期”，这里统一映射成 request_time。
if ("request_time" not in raw_df.columns) and ("apply_time" in raw_df.columns):
    raw_df["request_time"] = raw_df["apply_time"]
    print("[INFO] mapped apply_time -> request_time")

# 大白话：把截止日期统一解析成 pandas datetime（解析失败 -> NaT）。
raw_df["request_time"] = pd.to_datetime(raw_df.get("request_time"), errors="coerce")

# 大白话：我们需要每条样本至少有 apply_id、request_time、response_body。
# 大白话：执行下一行代码：required_cols = {\"apply_id\", \"request_time\", \"response_body\"}
required_cols = {"apply_id", "request_time", "response_body"}
# 大白话：执行下一行代码：missing_cols = sorted(list(required_cols - set(raw_df.columns)))
missing_cols = sorted(list(required_cols - set(raw_df.columns)))
# 大白话：执行下一行代码：if missing_cols:
if missing_cols:
    # 大白话：执行下一行代码：raise ValueError(f\"原始数据缺少关键列：{missing_cols}\")
    raise ValueError(f"原始数据缺少关键列：{missing_cols}")

# 大白话：把 request_time 统一解析成 pandas datetime；解析失败就变 NaT。
# 大白话：执行下一行代码：raw_df[\"request_time\"] = pd.to_datetime(raw_df[\"request_time\"], errors=\"coerce\")
raw_df["request_time"] = pd.to_datetime(raw_df["request_time"], errors="coerce")

# =========================
# 2) 读取 Excel（只用于核对，不产出文件）
# =========================
# 大白话：执行下一行代码：excel_feature_cnt = None
excel_feature_cnt = None
# 大白话：执行下一行代码：if XLSX_PATH.exists():
if XLSX_PATH.exists():
    # 大白话：执行下一行代码：try:
    try:
        # 大白话：执行下一行代码：xdf = pd.read_excel(XLSX_PATH, sheet_name=0)
        xdf = pd.read_excel(XLSX_PATH, sheet_name=0)
        # 大白话：Excel 第一行是表头，所以特征条数=有效行数。
        # 大白话：执行下一行代码：excel_feature_cnt = int(len(xdf))
        excel_feature_cnt = int(len(xdf))
    # 大白话：执行下一行代码：except Exception:
    except Exception:
        # 大白话：执行下一行代码：excel_feature_cnt = None
        excel_feature_cnt = None

# 大白话：执行下一行代码：print(\"raw_df shape:\", raw_df.shape)
print("raw_df shape:", raw_df.shape)
# 大白话：执行下一行代码：print(\"excel_feature_cnt (if readable):\", excel_feature_cnt)
print("excel_feature_cnt (if readable):", excel_feature_cnt)


[INFO] loaded pickle: cdc_pickle_pass_fpd7.pkl
[INFO] mapped apply_time -> request_time
raw_df shape: (12546, 12)
excel_feature_cnt (if readable): None


In [6]:
"""大白话：这个代码格负责“把 JSON 解析成可计算的明细表”。

我们会把每个 apply_id 的 response_body 解析出来，并把 4 个板块各自平铺成 DataFrame：
- consultas_df：每一行是一条查询记录（带 apply_id、cutoff=request_time）
- creditos_df：每一行是一条信贷账户记录（带 apply_id、cutoff=request_time）
- empleos_df：每一行是一条工作记录（带 apply_id、cutoff=request_time）
- domicilios_df：每一行是一条住址记录（带 apply_id、cutoff=request_time）

注意：这里只做解析/平铺/类型标准化，不做任何聚合特征。
"""

# =========================
# 1) 工具：安全解析 JSON
# =========================

# 大白话：执行下一行代码：def safe_json_loads(x: object) -> dict:
def safe_json_loads(x: object) -> dict:
    """把 response_body 安全解析成 dict；失败/为空返回空 dict。"""
    # 大白话：执行下一行代码：if x is None:
    if x is None:
        # 大白话：执行下一行代码：return {}
        return {}
    # 大白话：执行下一行代码：if isinstance(x, dict):
    if isinstance(x, dict):
        # 大白话：执行下一行代码：return x
        return x
    # 大白话：执行下一行代码：if not isinstance(x, str):
    if not isinstance(x, str):
        # 大白话：执行下一行代码：return {}
        return {}
    # 大白话：执行下一行代码：s = x.strip()
    s = x.strip()
    # 大白话：执行下一行代码：if not s:
    if not s:
        # 大白话：执行下一行代码：return {}
        return {}
    # 大白话：执行下一行代码：try:
    try:
        # 大白话：执行下一行代码：obj = json.loads(s)
        obj = json.loads(s)
        # 大白话：执行下一行代码：return obj if isinstance(obj, dict) else {}
        return obj if isinstance(obj, dict) else {}
    # 大白话：执行下一行代码：except Exception:
    except Exception:
        # 大白话：执行下一行代码：return {}
        return {}


# =========================
# 2) 平铺 4 个板块
# =========================
# 大白话：执行下一行代码：consultas_rows = []
consultas_rows = []
# 大白话：执行下一行代码：creditos_rows = []
creditos_rows = []
# 大白话：执行下一行代码：empleos_rows = []
empleos_rows = []
# 大白话：执行下一行代码：domicilios_rows = []
domicilios_rows = []

# 大白话：执行下一行代码：for _, r in raw_df[[\"apply_id\", \"request_time\", \"response_body\"]].iterrows():
for _, r in raw_df[["apply_id", "request_time", "response_body"]].iterrows():
    # 大白话：执行下一行代码：apply_id = r[\"apply_id\"]
    apply_id = r["apply_id"]
    # 大白话：执行下一行代码：cutoff = r[\"request_time\"]
    cutoff = r["request_time"]
    # 大白话：执行下一行代码：obj = safe_json_loads(r[\"response_body\"])
    obj = safe_json_loads(r["response_body"])

    # consultas
    # 大白话：执行下一行代码：consultas = obj.get(\"consultas\")
    consultas = obj.get("consultas")
    # 大白话：执行下一行代码：if isinstance(consultas, list):
    if isinstance(consultas, list):
        # 大白话：执行下一行代码：for it in consultas:
        for it in consultas:
            # 大白话：执行下一行代码：if not isinstance(it, dict):
            if not isinstance(it, dict):
                # 大白话：执行下一行代码：continue
                continue
            # 大白话：执行下一行代码：consultas_rows.append(
            consultas_rows.append(
                # 大白话：执行下一行代码：{
                {
                    # 大白话：执行下一行代码：\"apply_id\": apply_id,
                    "apply_id": apply_id,
                    # 大白话：执行下一行代码：\"cutoff\": cutoff,
                    "cutoff": cutoff,
                    # 大白话：执行下一行代码：\"fechaConsulta\": it.get(\"fechaConsulta\"),
                    "fechaConsulta": it.get("fechaConsulta"),
                    # 大白话：执行下一行代码：\"importeCredito\": it.get(\"importeCredito\"),
                    "importeCredito": it.get("importeCredito"),
                    # 大白话：执行下一行代码：\"tipoCredito\": it.get(\"tipoCredito\"),
                    "tipoCredito": it.get("tipoCredito"),
                # 大白话：执行下一行代码：}
                }
            # 大白话：执行下一行代码：)
            )

    # creditos
    # 大白话：执行下一行代码：creditos = obj.get(\"creditos\")
    creditos = obj.get("creditos")
    # 大白话：执行下一行代码：if isinstance(creditos, list):
    if isinstance(creditos, list):
        # 大白话：执行下一行代码：for it in creditos:
        for it in creditos:
            # 大白话：执行下一行代码：if not isinstance(it, dict):
            if not isinstance(it, dict):
                # 大白话：执行下一行代码：continue
                continue
            # 大白话：执行下一行代码：creditos_rows.append(
            creditos_rows.append(
                # 大白话：执行下一行代码：{
                {
                    # 大白话：执行下一行代码：\"apply_id\": apply_id,
                    "apply_id": apply_id,
                    # 大白话：执行下一行代码：\"cutoff\": cutoff,
                    "cutoff": cutoff,
                    # 大白话：执行下一行代码：\"fechaReporte\": it.get(\"fechaReporte\"),
                    "fechaReporte": it.get("fechaReporte"),
                    # 大白话：执行下一行代码：\"fechaCierreCuenta\": it.get(\"fechaCierreCuenta\"),
                    "fechaCierreCuenta": it.get("fechaCierreCuenta"),
                    # 大白话：执行下一行代码：\"fechaAperturaCuenta\": it.get(\"fechaAperturaCuenta\"),
                    "fechaAperturaCuenta": it.get("fechaAperturaCuenta"),
                    # 大白话：执行下一行代码：\"tipoResponsabilidad\": it.get(\"tipoResponsabilidad\"),
                    "tipoResponsabilidad": it.get("tipoResponsabilidad"),
                    # 大白话：执行下一行代码：\"saldoActual\": it.get(\"saldoActual\"),
                    "saldoActual": it.get("saldoActual"),
                    # 大白话：执行下一行代码：\"limiteCredito\": it.get(\"limiteCredito\"),
                    "limiteCredito": it.get("limiteCredito"),
                    # 大白话：执行下一行代码：\"frecuenciaPagos\": it.get(\"frecuenciaPagos\"),
                    "frecuenciaPagos": it.get("frecuenciaPagos"),
                    # 大白话：执行下一行代码：\"tipoCuenta\": it.get(\"tipoCuenta\"),
                    "tipoCuenta": it.get("tipoCuenta"),
                    # 大白话：执行下一行代码：\"tipoCredito\": it.get(\"tipoCredito\"),
                    "tipoCredito": it.get("tipoCredito"),
                    # 大白话：执行下一行代码：\"fechaPeorAtraso\": it.get(\"fechaPeorAtraso\"),
                    "fechaPeorAtraso": it.get("fechaPeorAtraso"),
                    # 大白话：执行下一行代码：\"fechaUltimoPago\": it.get(\"fechaUltimoPago\"),
                    "fechaUltimoPago": it.get("fechaUltimoPago"),
                    # 大白话：执行下一行代码：\"fechaUltimaCompra\": it.get(\"fechaUltimaCompra\"),
                    "fechaUltimaCompra": it.get("fechaUltimaCompra"),
                    # 大白话：执行下一行代码：\"fechaActualizacion\": it.get(\"fechaActualizacion\"),
                    "fechaActualizacion": it.get("fechaActualizacion"),
                    # 大白话：执行下一行代码：\"ultimaFechaSaldoCero\": it.get(\"ultimaFechaSaldoCero\"),
                    "ultimaFechaSaldoCero": it.get("ultimaFechaSaldoCero"),
                    # 大白话：执行下一行代码：\"clavePrevencion\": it.get(\"clavePrevencion\"),
                    "clavePrevencion": it.get("clavePrevencion"),
                    # 大白话：执行下一行代码：\"claveUnidadMonetaria\": it.get(\"claveUnidadMonetaria\"),
                    "claveUnidadMonetaria": it.get("claveUnidadMonetaria"),
                    # 大白话：执行下一行代码：\"numeroPagos\": it.get(\"numeroPagos\"),
                    "numeroPagos": it.get("numeroPagos"),
                    # 大白话：执行下一行代码：\"peorAtraso\": it.get(\"peorAtraso\"),
                    "peorAtraso": it.get("peorAtraso"),
                    # 大白话：执行下一行代码：\"registroImpugnado\": it.get(\"registroImpugnado\"),
                    "registroImpugnado": it.get("registroImpugnado"),
                    # 大白话：执行下一行代码：\"totalPagosReportados\": it.get(\"totalPagosReportados\"),
                    "totalPagosReportados": it.get("totalPagosReportados"),
                    # 大白话：执行下一行代码：\"saldoVencido\": it.get(\"saldoVencido\"),
                    "saldoVencido": it.get("saldoVencido"),
                # 大白话：执行下一行代码：}
                }
            # 大白话：执行下一行代码：)
            )

    # empleos
    # 大白话：执行下一行代码：empleos = obj.get(\"empleos\")
    empleos = obj.get("empleos")
    # 大白话：执行下一行代码：if isinstance(empleos, list):
    if isinstance(empleos, list):
        # 大白话：执行下一行代码：for it in empleos:
        for it in empleos:
            # 大白话：执行下一行代码：if not isinstance(it, dict):
            if not isinstance(it, dict):
                # 大白话：执行下一行代码：continue
                continue
            # 大白话：执行下一行代码：empleos_rows.append(
            empleos_rows.append(
                # 大白话：执行下一行代码：{
                {
                    # 大白话：执行下一行代码：\"apply_id\": apply_id,
                    "apply_id": apply_id,
                    # 大白话：执行下一行代码：\"cutoff\": cutoff,
                    "cutoff": cutoff,
                    # 大白话：执行下一行代码：\"salarioMensual\": it.get(\"salarioMensual\"),
                    "salarioMensual": it.get("salarioMensual"),
                    # 大白话：执行下一行代码：\"fechaContratacion\": it.get(\"fechaContratacion\"),
                    "fechaContratacion": it.get("fechaContratacion"),
                    # 大白话：执行下一行代码：\"fechaUltimoDiaEmpleo\": it.get(\"fechaUltimoDiaEmpleo\"),
                    "fechaUltimoDiaEmpleo": it.get("fechaUltimoDiaEmpleo"),
                    # 大白话：执行下一行代码：\"fechaVerificacionEmpleo\": it.get(\"fechaVerificacionEmpleo\"),
                    "fechaVerificacionEmpleo": it.get("fechaVerificacionEmpleo"),
                    # 大白话：执行下一行代码：\"nombreEmpresa\": it.get(\"nombreEmpresa\"),
                    "nombreEmpresa": it.get("nombreEmpresa"),
                # 大白话：执行下一行代码：}
                }
            # 大白话：执行下一行代码：)
            )

    # domicilios
    # 大白话：执行下一行代码：domicilios = obj.get(\"domicilios\")
    domicilios = obj.get("domicilios")
    # 大白话：执行下一行代码：if isinstance(domicilios, list):
    if isinstance(domicilios, list):
        # 大白话：执行下一行代码：for it in domicilios:
        for it in domicilios:
            # 大白话：执行下一行代码：if not isinstance(it, dict):
            if not isinstance(it, dict):
                # 大白话：执行下一行代码：continue
                continue
            # 大白话：执行下一行代码：domicilios_rows.append(
            domicilios_rows.append(
                # 大白话：执行下一行代码：{
                {
                    # 大白话：执行下一行代码：\"apply_id\": apply_id,
                    "apply_id": apply_id,
                    # 大白话：执行下一行代码：\"cutoff\": cutoff,
                    "cutoff": cutoff,
                    # 大白话：执行下一行代码：\"fechaResidencia\": it.get(\"fechaResidencia\"),
                    "fechaResidencia": it.get("fechaResidencia"),
                    # 大白话：执行下一行代码：\"estado\": it.get(\"estado\"),
                    "estado": it.get("estado"),
                    # 大白话：执行下一行代码：\"direccion\": it.get(\"direccion\"),
                    "direccion": it.get("direccion"),
                # 大白话：执行下一行代码：}
                }
            # 大白话：执行下一行代码：)
            )


# =========================
# 3) 组装 DataFrame + 类型标准化
# =========================
# 大白话：执行下一行代码：consultas_df = pd.DataFrame(consultas_rows)
consultas_df = pd.DataFrame(consultas_rows)
# 大白话：执行下一行代码：creditos_df = pd.DataFrame(creditos_rows)
creditos_df = pd.DataFrame(creditos_rows)
# 大白话：执行下一行代码：empleos_df = pd.DataFrame(empleos_rows)
empleos_df = pd.DataFrame(empleos_rows)
# 大白话：执行下一行代码：domicilios_df = pd.DataFrame(domicilios_rows)
domicilios_df = pd.DataFrame(domicilios_rows)

# 大白话：统一日期字段解析（失败->NaT）。
# 大白话：执行下一行代码：if len(consultas_df):
if len(consultas_df):
    # 大白话：执行下一行代码：consultas_df[\"fechaConsulta\"] = pd.to_datetime(consultas_df[\"fechaConsulta\"], errors=\"coerce\")
    consultas_df["fechaConsulta"] = pd.to_datetime(consultas_df["fechaConsulta"], errors="coerce")
# 大白话：执行下一行代码：if len(creditos_df):
if len(creditos_df):
    # 大白话：执行下一行代码：for c in [
    for c in [
        # 大白话：执行下一行代码：\"fechaReporte\",
        "fechaReporte",
        # 大白话：执行下一行代码：\"fechaCierreCuenta\",
        "fechaCierreCuenta",
        # 大白话：执行下一行代码：\"fechaAperturaCuenta\",
        "fechaAperturaCuenta",
        # 大白话：执行下一行代码：\"fechaPeorAtraso\",
        "fechaPeorAtraso",
        # 大白话：执行下一行代码：\"fechaUltimoPago\",
        "fechaUltimoPago",
        # 大白话：执行下一行代码：\"fechaUltimaCompra\",
        "fechaUltimaCompra",
        # 大白话：执行下一行代码：\"fechaActualizacion\",
        "fechaActualizacion",
        # 大白话：执行下一行代码：\"ultimaFechaSaldoCero\",
        "ultimaFechaSaldoCero",
    # 大白话：执行下一行代码：]:
    ]:
        # 大白话：执行下一行代码：creditos_df[c] = pd.to_datetime(creditos_df[c], errors=\"coerce\")
        creditos_df[c] = pd.to_datetime(creditos_df[c], errors="coerce")
# 大白话：执行下一行代码：if len(empleos_df):
if len(empleos_df):
    # 大白话：执行下一行代码：for c in [\"fechaContratacion\", \"fechaUltimoDiaEmpleo\", \"fechaVerificacionEmpleo\"]:
    for c in ["fechaContratacion", "fechaUltimoDiaEmpleo", "fechaVerificacionEmpleo"]:
        # 大白话：执行下一行代码：empleos_df[c] = pd.to_datetime(empleos_df[c], errors=\"coerce\")
        empleos_df[c] = pd.to_datetime(empleos_df[c], errors="coerce")
# 大白话：执行下一行代码：if len(domicilios_df):
if len(domicilios_df):
    # 大白话：执行下一行代码：domicilios_df[\"fechaResidencia\"] = pd.to_datetime(domicilios_df[\"fechaResidencia\"], errors=\"coerce\")
    domicilios_df["fechaResidencia"] = pd.to_datetime(domicilios_df["fechaResidencia"], errors="coerce")

# 大白话：统一数值字段（解析失败->NaN）。
# 大白话：执行下一行代码：num_cols_creditos = [
num_cols_creditos = [
    # 大白话：执行下一行代码：\"saldoActual\",
    "saldoActual",
    # 大白话：执行下一行代码：\"limiteCredito\",
    "limiteCredito",
    # 大白话：执行下一行代码：\"numeroPagos\",
    "numeroPagos",
    # 大白话：执行下一行代码：\"peorAtraso\",
    "peorAtraso",
    # 大白话：执行下一行代码：\"totalPagosReportados\",
    "totalPagosReportados",
    # 大白话：执行下一行代码：\"saldoVencido\",
    "saldoVencido",
# 大白话：执行下一行代码：]
]
# 大白话：执行下一行代码：for c in num_cols_creditos:
for c in num_cols_creditos:
    # 大白话：执行下一行代码：if len(creditos_df) and c in creditos_df.columns:
    if len(creditos_df) and c in creditos_df.columns:
        # 大白话：执行下一行代码：creditos_df[c] = pd.to_numeric(creditos_df[c], errors=\"coerce\")
        creditos_df[c] = pd.to_numeric(creditos_df[c], errors="coerce")

# 大白话：执行下一行代码：if len(consultas_df) and \"importeCredito\" in consultas_df.columns:
if len(consultas_df) and "importeCredito" in consultas_df.columns:
    # 大白话：执行下一行代码：consultas_df[\"importeCredito\"] = pd.to_numeric(consultas_df[\"importeCredito\"], errors=\"coerce\")
    consultas_df["importeCredito"] = pd.to_numeric(consultas_df["importeCredito"], errors="coerce")

# 大白话：执行下一行代码：if len(empleos_df) and \"salarioMensual\" in empleos_df.columns:
if len(empleos_df) and "salarioMensual" in empleos_df.columns:
    # 大白话：执行下一行代码：empleos_df[\"salarioMensual\"] = pd.to_numeric(empleos_df[\"salarioMensual\"], errors=\"coerce\")
    empleos_df["salarioMensual"] = pd.to_numeric(empleos_df["salarioMensual"], errors="coerce")

# 大白话：统一枚举字段为大写字符串（缺失->""）。
# 大白话：执行下一行代码：for df_, col in [
for df_, col in [
    # 大白话：执行下一行代码：(consultas_df, \"tipoCredito\"),
    (consultas_df, "tipoCredito"),
    # 大白话：执行下一行代码：(creditos_df, \"tipoCredito\"),
    (creditos_df, "tipoCredito"),
    # 大白话：执行下一行代码：(creditos_df, \"tipoCuenta\"),
    (creditos_df, "tipoCuenta"),
    # 大白话：执行下一行代码：(creditos_df, \"tipoResponsabilidad\"),
    (creditos_df, "tipoResponsabilidad"),
    # 大白话：执行下一行代码：(creditos_df, \"frecuenciaPagos\"),
    (creditos_df, "frecuenciaPagos"),
    # 大白话：执行下一行代码：(creditos_df, \"clavePrevencion\"),
    (creditos_df, "clavePrevencion"),
    # 大白话：执行下一行代码：(creditos_df, \"claveUnidadMonetaria\"),
    (creditos_df, "claveUnidadMonetaria"),
    # 大白话：执行下一行代码：(creditos_df, \"registroImpugnado\"),
    (creditos_df, "registroImpugnado"),
# 大白话：执行下一行代码：]:
]:
    # 大白话：执行下一行代码：if len(df_) and col in df_.columns:
    if len(df_) and col in df_.columns:
        # 大白话：执行下一行代码：df_[col] = df_[col].fillna(\"\").astype(str).str.strip().str.upper()
        df_[col] = df_[col].fillna("").astype(str).str.strip().str.upper()

# 大白话：执行下一行代码：if len(domicilios_df):
if len(domicilios_df):
    # 大白话：执行下一行代码：domicilios_df[\"estado\"] = domicilios_df[\"estado\"].fillna(\"\").astype(str).str.strip().str.upper()
    domicilios_df["estado"] = domicilios_df["estado"].fillna("").astype(str).str.strip().str.upper()
    # 大白话：执行下一行代码：domicilios_df[\"direccion\"] = domicilios_df[\"direccion\"].fillna(\"\").astype(str).str.strip()
    domicilios_df["direccion"] = domicilios_df["direccion"].fillna("").astype(str).str.strip()

# 大白话：执行下一行代码：print(\"consultas_df:\", consultas_df.shape)
print("consultas_df:", consultas_df.shape)
# 大白话：执行下一行代码：print(\"creditos_df:\", creditos_df.shape)
print("creditos_df:", creditos_df.shape)
# 大白话：执行下一行代码：print(\"empleos_df:\", empleos_df.shape)
print("empleos_df:", empleos_df.shape)
# 大白话：执行下一行代码：print(\"domicilios_df:\", domicilios_df.shape)
print("domicilios_df:", domicilios_df.shape)


consultas_df: (412045, 5)
creditos_df: (697815, 23)
empleos_df: (5188, 7)
domicilios_df: (37272, 5)


In [7]:
"""大白话：这个代码格负责“实现衍生逻辑函数”。

我们把 Excel 里的规则写成一个函数：
- 输入：某个 apply_id 对应的 cutoff（截止日期）、该 apply_id 的 consultas/creditos/empleos/domicilios 明细
- 输出：一个 dict（key=特征名，value=特征值）

统一约定：
- 计数类：没有记录就返回 0。
- max/mean/ratio 等：当分母为 0 或没有有效样本时返回 -999。
- 所有 float 输出最后保留 2 位小数（与 Excel 表述保持一致）。
"""

# 大白话：执行下一行代码：WINDOWS = [30, 60, 90, 180, 360, 720]
WINDOWS = [30, 60, 90, 180, 360, 720]
# 大白话：执行下一行代码：SENTINEL = -999.0
SENTINEL = -999.0


# 大白话：执行下一行代码：def days_between(cutoff: pd.Timestamp, dt: pd.Timestamp) -> float:
def days_between(cutoff: pd.Timestamp, dt: pd.Timestamp) -> float:
    """cutoff-dt 的天数（可带小数）；任一为空返回 NaN。"""
    # 大白话：执行下一行代码：if pd.isna(cutoff) or pd.isna(dt):
    if pd.isna(cutoff) or pd.isna(dt):
        # 大白话：执行下一行代码：return np.nan
        return np.nan
    # 大白话：执行下一行代码：return (cutoff - dt).total_seconds() / 86400.0
    return (cutoff - dt).total_seconds() / 86400.0


# 大白话：执行下一行代码：def in_cutoff_window(cutoff: pd.Timestamp, dt: pd.Timestamp, n_days: int) -> bool:
def in_cutoff_window(cutoff: pd.Timestamp, dt: pd.Timestamp, n_days: int) -> bool:
    """判断 dt 是否在 [cutoff-n_days, cutoff] 内（含边界）。"""
    # 大白话：执行下一行代码：d = days_between(cutoff, dt)
    d = days_between(cutoff, dt)
    # 大白话：执行下一行代码：if np.isnan(d):
    if np.isnan(d):
        # 大白话：执行下一行代码：return False
        return False
    # 大白话：执行下一行代码：return (d >= 0) and (d <= float(n_days))
    return (d >= 0) and (d <= float(n_days))


# 大白话：执行下一行代码：def safe_div(num: float, den: float) -> float:
def safe_div(num: float, den: float) -> float:
    """安全除法：den<=0 或 nan -> -999。"""
    # 大白话：执行下一行代码：if den is None or (isinstance(den, float) and np.isnan(den)):
    if den is None or (isinstance(den, float) and np.isnan(den)):
        # 大白话：执行下一行代码：return SENTINEL
        return SENTINEL
    # 大白话：执行下一行代码：if den == 0:
    if den == 0:
        # 大白话：执行下一行代码：return SENTINEL
        return SENTINEL
    # 大白话：执行下一行代码：return float(num) / float(den)
    return float(num) / float(den)


# 大白话：执行下一行代码：def mode_code_to_num(code: str) -> float:
def mode_code_to_num(code: str) -> float:
    """把 frecuenciaPagos 的众数 code 映射成数字（按 Excel 映射）。"""
    # 大白话：执行下一行代码：mp = {\"B\": 1, \"M\": 2, \"Q\": 3, \"A\": 4, \"D\": 5, \"T\": 6, \"S\": 7, \"E\": 8, \"C\": 9, \"U\": 10, \"R\": 11}
    mp = {"B": 1, "M": 2, "Q": 3, "A": 4, "D": 5, "T": 6, "S": 7, "E": 8, "C": 9, "U": 10, "R": 11}
    # 大白话：执行下一行代码：return float(mp.get(code, SENTINEL))
    return float(mp.get(code, SENTINEL))


# 大白话：执行下一行代码：def pick_current_salary(emp: pd.DataFrame, cutoff: pd.Timestamp) -> float:
def pick_current_salary(emp: pd.DataFrame, cutoff: pd.Timestamp) -> float:
    """当前月薪：优先取“未离职”的工作中，fechaContratacion 最大的那条 salarioMensual。否则取 fechaContratacion 最大的那条。"""
    # 大白话：执行下一行代码：if emp is None or len(emp) == 0:
    if emp is None or len(emp) == 0:
        # 大白话：执行下一行代码：return np.nan
        return np.nan
    # 大白话：执行下一行代码：df = emp.copy()
    df = emp.copy()
    # 大白话：执行下一行代码：df = df[df[\"salarioMensual\"].notna()]
    df = df[df["salarioMensual"].notna()]
    # 大白话：执行下一行代码：if len(df) == 0:
    if len(df) == 0:
        # 大白话：执行下一行代码：return np.nan
        return np.nan

    # 未离职：fechaUltimoDiaEmpleo 为空
    # 大白话：执行下一行代码：cur = df[df[\"fechaUltimoDiaEmpleo\"].isna()]
    cur = df[df["fechaUltimoDiaEmpleo"].isna()]
    # 大白话：执行下一行代码：cand = cur if len(cur) else df
    cand = cur if len(cur) else df
    # 大白话：执行下一行代码：cand = cand.sort_values(by=[\"fechaContratacion\"], ascending=[False])
    cand = cand.sort_values(by=["fechaContratacion"], ascending=[False])
    # 大白话：执行下一行代码：v = cand.iloc[0][\"salarioMensual\"]
    v = cand.iloc[0]["salarioMensual"]
    # 大白话：执行下一行代码：return float(v) if pd.notna(v) else np.nan
    return float(v) if pd.notna(v) else np.nan


# 大白话：执行下一行代码：def compute_job_months(start: pd.Timestamp, end: pd.Timestamp) -> float:
def compute_job_months(start: pd.Timestamp, end: pd.Timestamp) -> float:
    """工作持续月数：天数/30。"""
    # 大白话：执行下一行代码：if pd.isna(start) or pd.isna(end):
    if pd.isna(start) or pd.isna(end):
        # 大白话：执行下一行代码：return np.nan
        return np.nan
    # 大白话：执行下一行代码：return float((end - start).total_seconds() / 86400.0) / 30.0
    return float((end - start).total_seconds() / 86400.0) / 30.0


# 大白话：执行下一行代码：def compute_res_months(start: pd.Timestamp, cutoff: pd.Timestamp) -> float:
def compute_res_months(start: pd.Timestamp, cutoff: pd.Timestamp) -> float:
    """居住月数：天数/30。"""
    # 大白话：执行下一行代码：if pd.isna(start) or pd.isna(cutoff):
    if pd.isna(start) or pd.isna(cutoff):
        # 大白话：执行下一行代码：return np.nan
        return np.nan
    # 大白话：执行下一行代码：return float((cutoff - start).total_seconds() / 86400.0) / 30.0
    return float((cutoff - start).total_seconds() / 86400.0) / 30.0


# 大白话：执行下一行代码：def derive_one_apply(
def derive_one_apply(
    # 大白话：执行下一行代码：apply_id: object,
    apply_id: object,
    # 大白话：执行下一行代码：cutoff: pd.Timestamp,
    cutoff: pd.Timestamp,
    # 大白话：执行下一行代码：cst: pd.DataFrame,
    cst: pd.DataFrame,
    # 大白话：执行下一行代码：cre: pd.DataFrame,
    cre: pd.DataFrame,
    # 大白话：执行下一行代码：emp: pd.DataFrame,
    emp: pd.DataFrame,
    # 大白话：执行下一行代码：dom: pd.DataFrame,
    dom: pd.DataFrame,
# 大白话：执行下一行代码：) -> dict:
) -> dict:
    """按 Excel 口径衍生一条样本的全部 BOSS 特征。"""

    # 大白话：执行下一行代码：out: dict[str, float] = {}
    out: dict[str, float] = {}

    # ========== 防穿越：把“晚于 cutoff(request_time) 的日期”统一视为无效（置为 NaT）=========
    # 大白话：只允许使用截止日及之前的信息；如果某条明细的日期字段 > cutoff，就属于未来信息，必须置空。
    # 大白话：执行下一行代码：if pd.notna(cutoff):
    if pd.notna(cutoff):
        # 大白话：执行下一行代码：for _df, _cols in [...]:
        for _df, _cols in [
            (
                cre,
                [
                    "fechaReporte",
                    "fechaCierreCuenta",
                    "fechaAperturaCuenta",
                    "fechaPeorAtraso",
                    "fechaUltimoPago",
                    "fechaUltimaCompra",
                    "fechaActualizacion",
                    "ultimaFechaSaldoCero",
                ],
            ),
            (cst, ["fechaConsulta"]),
            (emp, ["fechaContratacion", "fechaUltimoDiaEmpleo", "fechaVerificacionEmpleo"]),
            (dom, ["fechaResidencia"]),
        ]:
            # 大白话：执行下一行代码：for _c in _cols:
            for _c in _cols:
                # 大白话：执行下一行代码：if _c in _df.columns:
                if _c in _df.columns:
                    # 大白话：执行下一行代码：_df.loc[_df[_c].notna() & (_df[_c] > cutoff), _c] = pd.NaT
                    _df.loc[_df[_c].notna() & (_df[_c] > cutoff), _c] = pd.NaT

    # -------------------------
    # A) creditos：账户数/状态
    # -------------------------
    # 大白话：执行下一行代码：total_accounts = int(len(cre)) if cre is not None else 0
    total_accounts = int(len(cre)) if cre is not None else 0
    # 大白话：执行下一行代码：out[\"boss_total_accounts\"] = float(total_accounts)
    out["boss_total_accounts"] = float(total_accounts)

    # 大白话：执行下一行代码：if total_accounts == 0:
    if total_accounts == 0:
        # 大白话：creditos 为空时，计数类=0；需要除法/均值/最大值的特征后面会给 -999。
        # 大白话：执行下一行代码：pass
        pass

    # 已关闭/开放
    # 大白话：执行下一行代码：closed_cnt = int(cre[\"fechaCierreCuenta\"].notna().sum()) if total_accounts else 0
    closed_cnt = int(cre["fechaCierreCuenta"].notna().sum()) if total_accounts else 0
    # 大白话：执行下一行代码：open_cnt = int(cre[\"fechaCierreCuenta\"].isna().sum()) if total_accounts else 0
    open_cnt = int(cre["fechaCierreCuenta"].isna().sum()) if total_accounts else 0
    # 大白话：执行下一行代码：out[\"boss_closed_accounts_cnt\"] = float(closed_cnt)
    out["boss_closed_accounts_cnt"] = float(closed_cnt)
    # 大白话：执行下一行代码：out[\"boss_open_accounts_cnt\"] = float(open_cnt)
    out["boss_open_accounts_cnt"] = float(open_cnt)

    # 零余额
    # 大白话：执行下一行代码：zero_balance_cnt = int((cre[\"saldoActual\"].fillna(np.nan) == 0).sum()) if total_accounts else 0
    zero_balance_cnt = int((cre["saldoActual"].fillna(np.nan) == 0).sum()) if total_accounts else 0
    # 大白话：执行下一行代码：out[\"boss_zero_balance_accounts_cnt\"] = float(zero_balance_cnt)
    out["boss_zero_balance_accounts_cnt"] = float(zero_balance_cnt)

    # 责任类型：I/M/A
    # 大白话：执行下一行代码：tipo = cre[\"tipoResponsabilidad\"].fillna(\"\") if total_accounts else pd.Series([], dtype=str)
    tipo = cre["tipoResponsabilidad"].fillna("") if total_accounts else pd.Series([], dtype=str)
    # 大白话：执行下一行代码：out[\"boss_respons_individual_cnt\"] = float(int((tipo == \"I\").sum())) if total_accounts else 0.0
    out["boss_respons_individual_cnt"] = float(int((tipo == "I").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_respons_joint_cnt\"] = float(int((tipo == \"M\").sum())) if total_accounts else 0.0
    out["boss_respons_joint_cnt"] = float(int((tipo == "M").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_respons_guarantor_a_cnt\"] = float(int((tipo == \"A\").sum())) if total_accounts else 0.0
    out["boss_respons_guarantor_a_cnt"] = float(int((tipo == "A").sum())) if total_accounts else 0.0

    # 余额汇总
    # 大白话：执行下一行代码：saldo = cre[\"saldoActual\"].fillna(0.0) if total_accounts else pd.Series([], dtype=float)
    saldo = cre["saldoActual"].fillna(0.0) if total_accounts else pd.Series([], dtype=float)
    # 大白话：执行下一行代码：out[\"boss_individual_balance_sum\"] = float(saldo[tipo == \"I\"].sum()) if total_accounts else 0.0
    out["boss_individual_balance_sum"] = float(saldo[tipo == "I"].sum()) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_joint_balance_sum\"] = float(saldo[tipo == \"M\"].sum()) if total_accounts else 0.0
    out["boss_joint_balance_sum"] = float(saldo[tipo == "M"].sum()) if total_accounts else 0.0

    # 账户类型/信贷类型数量
    # 大白话：执行下一行代码：out[\"boss_tipo_cuenta_nunique\"] = float(cre[\"tipoCuenta\"].replace(\"\", np.nan).dropna().nunique()) if total_accounts else 0.0
    out["boss_tipo_cuenta_nunique"] = float(cre["tipoCuenta"].replace("", np.nan).dropna().nunique()) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_tipo_credito_nunique\"] = float(cre[\"tipoCredito\"].replace(\"\", np.nan).dropna().nunique()) if total_accounts else 0.0
    out["boss_tipo_credito_nunique"] = float(cre["tipoCredito"].replace("", np.nan).dropna().nunique()) if total_accounts else 0.0

    # 币种账户数
    # 大白话：执行下一行代码：cu = cre[\"claveUnidadMonetaria\"].fillna(\"\") if total_accounts else pd.Series([], dtype=str)
    cu = cre["claveUnidadMonetaria"].fillna("") if total_accounts else pd.Series([], dtype=str)
    # 大白话：执行下一行代码：out[\"boss_currency_mx_cnt\"] = float(int((cu == \"MX\").sum())) if total_accounts else 0.0
    out["boss_currency_mx_cnt"] = float(int((cu == "MX").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_currency_us_cnt\"] = float(int((cu == \"US\").sum())) if total_accounts else 0.0
    out["boss_currency_us_cnt"] = float(int((cu == "US").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_currency_ud_cnt\"] = float(int((cu == \"UD\").sum())) if total_accounts else 0.0
    out["boss_currency_ud_cnt"] = float(int((cu == "UD").sum())) if total_accounts else 0.0

    # 预警/争议
    # 大白话：执行下一行代码：out[\"boss_has_warning_accounts_cnt\"] = float(int((cre[\"clavePrevencion\"].fillna(\"\") != \"\").sum())) if total_accounts else 0.0
    out["boss_has_warning_accounts_cnt"] = float(int((cre["clavePrevencion"].fillna("") != "").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_disputed_accounts_cnt\"] = float(int((cre[\"registroImpugnado\"].fillna(\"\") == \"1\").sum())) if total_accounts else 0.0
    out["boss_disputed_accounts_cnt"] = float(int((cre["registroImpugnado"].fillna("") == "1").sum())) if total_accounts else 0.0

    # 最差逾期状态（peorAtraso max）
    # 大白话：执行下一行代码：peor = cre[\"peorAtraso\"].dropna() if total_accounts else pd.Series([], dtype=float)
    peor = cre["peorAtraso"].dropna() if total_accounts else pd.Series([], dtype=float)
    # 大白话：执行下一行代码：out[\"boss_worst_arrears_max\"] = float(peor.max()) if len(peor) else SENTINEL
    out["boss_worst_arrears_max"] = float(peor.max()) if len(peor) else SENTINEL

    # 还款期数（numeroPagos）
    # 大白话：执行下一行代码：npagos = cre[\"numeroPagos\"].dropna() if total_accounts else pd.Series([], dtype=float)
    npagos = cre["numeroPagos"].dropna() if total_accounts else pd.Series([], dtype=float)
    # 大白话：执行下一行代码：out[\"boss_num_pagos_mean\"] = float(npagos.mean()) if len(npagos) else SENTINEL
    out["boss_num_pagos_mean"] = float(npagos.mean()) if len(npagos) else SENTINEL
    # 大白话：执行下一行代码：out[\"boss_num_pagos_sum\"] = float(npagos.sum()) if total_accounts else 0.0
    out["boss_num_pagos_sum"] = float(npagos.sum()) if total_accounts else 0.0

    # 已报告还款次数
    # 大白话：执行下一行代码：tpr = cre[\"totalPagosReportados\"].dropna() if total_accounts else pd.Series([], dtype=float)
    tpr = cre["totalPagosReportados"].dropna() if total_accounts else pd.Series([], dtype=float)
    # 大白话：执行下一行代码：out[\"boss_total_pagos_reportados_mean\"] = float(tpr.mean()) if len(tpr) else SENTINEL
    out["boss_total_pagos_reportados_mean"] = float(tpr.mean()) if len(tpr) else SENTINEL
    # 大白话：执行下一行代码：out[\"boss_total_pagos_reportados_sum\"] = float(tpr.sum()) if total_accounts else 0.0
    out["boss_total_pagos_reportados_sum"] = float(tpr.sum()) if total_accounts else 0.0

    # 逾期余额收入比
    # 大白话：执行下一行代码：current_salary = pick_current_salary(emp, cutoff)
    current_salary = pick_current_salary(emp, cutoff)
    # 大白话：执行下一行代码：overdue_sum = float(cre[\"saldoVencido\"].fillna(0.0).sum()) if total_accounts else 0.0
    overdue_sum = float(cre["saldoVencido"].fillna(0.0).sum()) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_overdue_to_income\"] = safe_div(overdue_sum, current_salary) if pd.notna(current_salary) else SENTINEL
    out["boss_overdue_to_income"] = safe_div(overdue_sum, current_salary) if pd.notna(current_salary) else SENTINEL

    # 负债收入比 / 还款收入比（按 Excel：都用 saldoActual 总和/当前月薪）
    # 大白话：执行下一行代码：total_balance_sum = float(cre[\"saldoActual\"].fillna(0.0).sum()) if total_accounts else 0.0
    total_balance_sum = float(cre["saldoActual"].fillna(0.0).sum()) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_dti_total_balance_to_income\"] = safe_div(total_balance_sum, current_salary) if pd.notna(current_salary) else SENTINEL
    out["boss_dti_total_balance_to_income"] = safe_div(total_balance_sum, current_salary) if pd.notna(current_salary) else SENTINEL
    # 大白话：执行下一行代码：out[\"boss_repay_to_income\"] = safe_div(total_balance_sum, current_salary) if pd.notna(current_salary) else SENTINEL
    out["boss_repay_to_income"] = safe_div(total_balance_sum, current_salary) if pd.notna(current_salary) else SENTINEL

    # 授信收入比
    # 大白话：执行下一行代码：total_limit_sum = float(cre[\"limiteCredito\"].fillna(0.0).sum()) if total_accounts else 0.0
    total_limit_sum = float(cre["limiteCredito"].fillna(0.0).sum()) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_limit_to_income\"] = safe_div(total_limit_sum, current_salary) if pd.notna(current_salary) else SENTINEL
    out["boss_limit_to_income"] = safe_div(total_limit_sum, current_salary) if pd.notna(current_salary) else SENTINEL

    # 还款频率众数映射
    # 大白话：执行下一行代码：fp = cre[\"frecuenciaPagos\"].replace(\"\", np.nan).dropna() if total_accounts else pd.Series([], dtype=str)
    fp = cre["frecuenciaPagos"].replace("", np.nan).dropna() if total_accounts else pd.Series([], dtype=str)
    # 大白话：执行下一行代码：if len(fp):
    if len(fp):
        # 大白话：执行下一行代码：mode = fp.value_counts().idxmax()
        mode = fp.value_counts().idxmax()
        # 大白话：执行下一行代码：out[\"boss_payment_freq_mode_num\"] = mode_code_to_num(str(mode))
        out["boss_payment_freq_mode_num"] = mode_code_to_num(str(mode))
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_payment_freq_mode_num\"] = SENTINEL
        out["boss_payment_freq_mode_num"] = SENTINEL

    # 还款频率账户数
    # 大白话：执行下一行代码：out[\"boss_pay_freq_weekly_cnt\"] = float(int((cre[\"frecuenciaPagos\"] == \"S\").sum())) if total_accounts else 0.0
    out["boss_pay_freq_weekly_cnt"] = float(int((cre["frecuenciaPagos"] == "S").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_pay_freq_biweekly_cnt\"] = float(int((cre[\"frecuenciaPagos\"].isin([\"Q\", \"C\"]).sum()))) if total_accounts else 0.0
    out["boss_pay_freq_biweekly_cnt"] = float(int((cre["frecuenciaPagos"].isin(["Q", "C"]).sum()))) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_pay_freq_monthly_cnt\"] = float(int((cre[\"frecuenciaPagos\"] == \"M\").sum())) if total_accounts else 0.0
    out["boss_pay_freq_monthly_cnt"] = float(int((cre["frecuenciaPagos"] == "M").sum())) if total_accounts else 0.0
    # 大白话：执行下一行代码：out[\"boss_pay_freq_semiannual_cnt\"] = float(int((cre[\"frecuenciaPagos\"] == \"E\").sum())) if total_accounts else 0.0
    out["boss_pay_freq_semiannual_cnt"] = float(int((cre["frecuenciaPagos"] == "E").sum())) if total_accounts else 0.0

    # 信用历史长度（月）：(最晚开户日 - 最早开户日)/30
    # 大白话：执行下一行代码：ap = cre[\"fechaAperturaCuenta\"].dropna() if total_accounts else pd.Series([], dtype=\"datetime64[ns]\")
    ap = cre["fechaAperturaCuenta"].dropna() if total_accounts else pd.Series([], dtype="datetime64[ns]")
    # 大白话：执行下一行代码：if len(ap) >= 1:
    if len(ap) >= 1:
        # 大白话：执行下一行代码：ap_min = ap.min()
        ap_min = ap.min()
        # 大白话：执行下一行代码：ap_max = ap.max()
        ap_max = ap.max()
        # 大白话：执行下一行代码：out[\"boss_credit_history_months\"] = float((ap_max - ap_min).total_seconds() / 86400.0) / 30.0
        out["boss_credit_history_months"] = float((ap_max - ap_min).total_seconds() / 86400.0) / 30.0
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_credit_history_months\"] = SENTINEL
        out["boss_credit_history_months"] = SENTINEL

    # 平均账龄（天）：mean(cutoff - fechaAperturaCuenta)
    # 大白话：执行下一行代码：if len(ap) >= 1 and pd.notna(cutoff):
    if len(ap) >= 1 and pd.notna(cutoff):
        # 大白话：执行下一行代码：ages = (cutoff - ap).dt.total_seconds() / 86400.0
        ages = (cutoff - ap).dt.total_seconds() / 86400.0
        # 大白话：执行下一行代码：ages = ages[(ages >= 0)]
        ages = ages[(ages >= 0)]
        # 大白话：执行下一行代码：out[\"boss_avg_account_age_days\"] = float(ages.mean()) if len(ages) else SENTINEL
        out["boss_avg_account_age_days"] = float(ages.mean()) if len(ages) else SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_avg_account_age_days\"] = SENTINEL
        out["boss_avg_account_age_days"] = SENTINEL

    # 最新/最老账户账龄（天/月）
    # 大白话：执行下一行代码：if len(ap) >= 1 and pd.notna(cutoff):
    if len(ap) >= 1 and pd.notna(cutoff):
        # 大白话：执行下一行代码：newest = ap.max()
        newest = ap.max()
        # 大白话：执行下一行代码：oldest = ap.min()
        oldest = ap.min()
        # 大白话：执行下一行代码：newest_days = days_between(cutoff, newest)
        newest_days = days_between(cutoff, newest)
        # 大白话：执行下一行代码：oldest_days = days_between(cutoff, oldest)
        oldest_days = days_between(cutoff, oldest)
        # 大白话：执行下一行代码：out[\"boss_newest_account_age_days\"] = float(newest_days) if not np.isnan(newest_days) else SENTINEL
        out["boss_newest_account_age_days"] = float(newest_days) if not np.isnan(newest_days) else SENTINEL
        # 大白话：执行下一行代码：out[\"boss_oldest_account_age_days\"] = float(oldest_days) if not np.isnan(oldest_days) else SENTINEL
        out["boss_oldest_account_age_days"] = float(oldest_days) if not np.isnan(oldest_days) else SENTINEL
        # 大白话：执行下一行代码：out[\"boss_oldest_account_age_months\"] = float(oldest_days) / 30.0 if not np.isnan(oldest_days) else SENTINEL
        out["boss_oldest_account_age_months"] = float(oldest_days) / 30.0 if not np.isnan(oldest_days) else SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_newest_account_age_days\"] = SENTINEL
        out["boss_newest_account_age_days"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_oldest_account_age_days\"] = SENTINEL
        out["boss_oldest_account_age_days"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_oldest_account_age_months\"] = SENTINEL
        out["boss_oldest_account_age_months"] = SENTINEL

    # 平均开户间隔（天）：相邻开户日期差的平均
    # 大白话：执行下一行代码：if len(ap) >= 2:
    if len(ap) >= 2:
        # 大白话：执行下一行代码：ap_sorted = ap.sort_values()
        ap_sorted = ap.sort_values()
        # 大白话：执行下一行代码：diffs = ap_sorted.diff().dropna().dt.total_seconds() / 86400.0
        diffs = ap_sorted.diff().dropna().dt.total_seconds() / 86400.0
        # 大白话：执行下一行代码：out[\"boss_avg_open_interval_days\"] = float(diffs.mean()) if len(diffs) else SENTINEL
        out["boss_avg_open_interval_days"] = float(diffs.mean()) if len(diffs) else SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_avg_open_interval_days\"] = SENTINEL
        out["boss_avg_open_interval_days"] = SENTINEL

    # 距最后一次事件天数：取对应日期的最大值
    # 大白话：执行下一行代码：def last_days(col: str) -> float:
    def last_days(col: str) -> float:
        # 大白话：执行下一行代码：if total_accounts == 0 or pd.isna(cutoff):
        if total_accounts == 0 or pd.isna(cutoff):
            # 大白话：执行下一行代码：return SENTINEL
            return SENTINEL
        # 大白话：执行下一行代码：s = cre[col].dropna()
        s = cre[col].dropna()
        # 大白话：执行下一行代码：if len(s) == 0:
        if len(s) == 0:
            # 大白话：执行下一行代码：return SENTINEL
            return SENTINEL
        # 大白话：执行下一行代码：return float(days_between(cutoff, s.max()))
        return float(days_between(cutoff, s.max()))

    # 大白话：执行下一行代码：out[\"boss_days_since_last_close\"] = last_days(\"fechaCierreCuenta\")
    out["boss_days_since_last_close"] = last_days("fechaCierreCuenta")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_report\"] = last_days(\"fechaReporte\")
    out["boss_days_since_last_report"] = last_days("fechaReporte")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_update\"] = last_days(\"fechaActualizacion\")
    out["boss_days_since_last_update"] = last_days("fechaActualizacion")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_zero_balance\"] = last_days(\"ultimaFechaSaldoCero\")
    out["boss_days_since_last_zero_balance"] = last_days("ultimaFechaSaldoCero")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_worst_arrears_date\"] = last_days(\"fechaPeorAtraso\")
    out["boss_days_since_last_worst_arrears_date"] = last_days("fechaPeorAtraso")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_payment\"] = last_days(\"fechaUltimoPago\")
    out["boss_days_since_last_payment"] = last_days("fechaUltimoPago")
    # 大白话：执行下一行代码：out[\"boss_days_since_last_purchase\"] = last_days(\"fechaUltimaCompra\")
    out["boss_days_since_last_purchase"] = last_days("fechaUltimaCompra")

    # -------------------------
    # B) creditos：窗口特征（用 fechaReporte 筛）
    # -------------------------
    # 大白话：执行下一行代码：if total_accounts and pd.notna(cutoff):
    if total_accounts and pd.notna(cutoff):
        # 大白话：执行下一行代码：report = cre[\"fechaReporte\"]
        report = cre["fechaReporte"]
        # 大白话：执行下一行代码：in_w = {w: cre[report.apply(lambda x: in_cutoff_window(cutoff, x, w))] for w in WINDOWS}
        in_w = {w: cre[report.apply(lambda x: in_cutoff_window(cutoff, x, w))] for w in WINDOWS}
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：in_w = {w: cre.iloc[0:0] for w in WINDOWS}
        in_w = {w: cre.iloc[0:0] for w in WINDOWS}

    # 大白话：执行下一行代码：for w in WINDOWS:
    for w in WINDOWS:
        # 大白话：执行下一行代码：dfw = in_w[w]
        dfw = in_w[w]
        # 大白话：执行下一行代码：w_prefix = f\"boss_w{w}d_\"
        w_prefix = f"boss_w{w}d_"

        # 窗口内活跃账户数：窗口内记录数（按 fechaReporte）
        # 大白话：执行下一行代码：out[w_prefix + \"active_accounts_cnt\"] = float(len(dfw))
        out[w_prefix + "active_accounts_cnt"] = float(len(dfw))

        # 窗口内关闭/新开：按 Excel 行为，这里理解为“窗口内记录中，关闭日期/开户日期存在的账户数”
        # 大白话：执行下一行代码：out[w_prefix + \"closed_accounts_cnt\"] = float(int(dfw[\"fechaCierreCuenta\"].notna().sum())) if len(dfw) else 0.0
        out[w_prefix + "closed_accounts_cnt"] = float(int(dfw["fechaCierreCuenta"].notna().sum())) if len(dfw) else 0.0
        # 大白话：执行下一行代码：out[w_prefix + \"opened_accounts_cnt\"] = float(int(dfw[\"fechaAperturaCuenta\"].notna().sum())) if len(dfw) else 0.0
        out[w_prefix + "opened_accounts_cnt"] = float(int(dfw["fechaAperturaCuenta"].notna().sum())) if len(dfw) else 0.0

        # 窗口内账户平均使用率：平均(saldoActual/limiteCredito)，仅限 limiteCredito>0
        # 大白话：执行下一行代码：if len(dfw):
        if len(dfw):
            # 大白话：执行下一行代码：lim = dfw[\"limiteCredito\"].fillna(0.0)
            lim = dfw["limiteCredito"].fillna(0.0)
            # 大白话：执行下一行代码：bal = dfw[\"saldoActual\"].fillna(0.0)
            bal = dfw["saldoActual"].fillna(0.0)
            # 大白话：执行下一行代码：mask = lim > 0
            mask = lim > 0
            # 大白话：执行下一行代码：util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan).dropna()
            util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan).dropna()
            # 大白话：执行下一行代码：out[w_prefix + \"avg_utilization\"] = float(util.mean()) if len(util) else SENTINEL
            out[w_prefix + "avg_utilization"] = float(util.mean()) if len(util) else SENTINEL
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[w_prefix + \"avg_utilization\"] = SENTINEL
            out[w_prefix + "avg_utilization"] = SENTINEL

        # 窗口内高使用率账户数：使用率>0.8
        # 大白话：执行下一行代码：if len(dfw):
        if len(dfw):
            # 大白话：执行下一行代码：lim = dfw[\"limiteCredito\"].fillna(0.0)
            lim = dfw["limiteCredito"].fillna(0.0)
            # 大白话：执行下一行代码：bal = dfw[\"saldoActual\"].fillna(0.0)
            bal = dfw["saldoActual"].fillna(0.0)
            # 大白话：执行下一行代码：mask = lim > 0
            mask = lim > 0
            # 大白话：执行下一行代码：util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan)
            util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan)
            # 大白话：执行下一行代码：out[w_prefix + \"high_util_cnt\"] = float(int((util > 0.8).sum()))
            out[w_prefix + "high_util_cnt"] = float(int((util > 0.8).sum()))
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[w_prefix + \"high_util_cnt\"] = 0.0
            out[w_prefix + "high_util_cnt"] = 0.0

        # 窗口内总使用率：sum(balance)/sum(limit)
        # 大白话：执行下一行代码：if len(dfw):
        if len(dfw):
            # 大白话：执行下一行代码：lim_sum = float(dfw[\"limiteCredito\"].fillna(0.0).sum())
            lim_sum = float(dfw["limiteCredito"].fillna(0.0).sum())
            # 大白话：执行下一行代码：bal_sum = float(dfw[\"saldoActual\"].fillna(0.0).sum())
            bal_sum = float(dfw["saldoActual"].fillna(0.0).sum())
            # 大白话：执行下一行代码：out[w_prefix + \"total_utilization\"] = safe_div(bal_sum, lim_sum)
            out[w_prefix + "total_utilization"] = safe_div(bal_sum, lim_sum)
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[w_prefix + \"total_utilization\"] = SENTINEL
            out[w_prefix + "total_utilization"] = SENTINEL

        # 窗口内信用卡使用率：tipoCredito==TC 的平均使用率
        # 大白话：执行下一行代码：df_tc = dfw[dfw[\"tipoCredito\"] == \"TC\"] if len(dfw) else dfw
        df_tc = dfw[dfw["tipoCredito"] == "TC"] if len(dfw) else dfw
        # 大白话：执行下一行代码：if len(df_tc):
        if len(df_tc):
            # 大白话：执行下一行代码：lim = df_tc[\"limiteCredito\"].fillna(0.0)
            lim = df_tc["limiteCredito"].fillna(0.0)
            # 大白话：执行下一行代码：bal = df_tc[\"saldoActual\"].fillna(0.0)
            bal = df_tc["saldoActual"].fillna(0.0)
            # 大白话：执行下一行代码：mask = lim > 0
            mask = lim > 0
            # 大白话：执行下一行代码：util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan).dropna()
            util = (bal[mask] / lim[mask]).replace([np.inf, -np.inf], np.nan).dropna()
            # 大白话：执行下一行代码：out[w_prefix + \"tc_avg_utilization\"] = float(util.mean()) if len(util) else SENTINEL
            out[w_prefix + "tc_avg_utilization"] = float(util.mean()) if len(util) else SENTINEL
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[w_prefix + \"tc_avg_utilization\"] = SENTINEL
            out[w_prefix + "tc_avg_utilization"] = SENTINEL

        # 窗口内授信收入比 / 负债收入比 / 还款收入比：窗口内 sum(limit)/salary, sum(balance)/salary
        # 大白话：执行下一行代码：if pd.notna(current_salary):
        if pd.notna(current_salary):
            # 大白话：执行下一行代码：out[w_prefix + \"limit_to_income\"] = safe_div(float(dfw[\"limiteCredito\"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
            out[w_prefix + "limit_to_income"] = safe_div(float(dfw["limiteCredito"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
            # 大白话：执行下一行代码：out[w_prefix + \"dti_balance_to_income\"] = safe_div(float(dfw[\"saldoActual\"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
            out[w_prefix + "dti_balance_to_income"] = safe_div(float(dfw["saldoActual"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
            # 大白话：执行下一行代码：out[w_prefix + \"repay_to_income\"] = safe_div(float(dfw[\"saldoActual\"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
            out[w_prefix + "repay_to_income"] = safe_div(float(dfw["saldoActual"].fillna(0.0).sum()), current_salary) if len(dfw) else safe_div(0.0, current_salary)
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[w_prefix + \"limit_to_income\"] = SENTINEL
            out[w_prefix + "limit_to_income"] = SENTINEL
            # 大白话：执行下一行代码：out[w_prefix + \"dti_balance_to_income\"] = SENTINEL
            out[w_prefix + "dti_balance_to_income"] = SENTINEL
            # 大白话：执行下一行代码：out[w_prefix + \"repay_to_income\"] = SENTINEL
            out[w_prefix + "repay_to_income"] = SENTINEL

    # 开户速度：近180天新开账户数/6（月）——这里用 fechaAperturaCuenta 与 cutoff 做差
    # 大白话：执行下一行代码：if total_accounts and pd.notna(cutoff):
    if total_accounts and pd.notna(cutoff):
        # 大白话：执行下一行代码：ap_days = (cutoff - cre[\"fechaAperturaCuenta\"]).dt.total_seconds() / 86400.0
        ap_days = (cutoff - cre["fechaAperturaCuenta"]).dt.total_seconds() / 86400.0
        # 大白话：执行下一行代码：new_180 = int(((ap_days >= 0) & (ap_days <= 180)).sum())
        new_180 = int(((ap_days >= 0) & (ap_days <= 180)).sum())
        # 大白话：执行下一行代码：out[\"boss_open_speed_per_month\"] = float(new_180) / 6.0
        out["boss_open_speed_per_month"] = float(new_180) / 6.0
        # 大白话：执行下一行代码：new_360 = int(((ap_days >= 0) & (ap_days <= 360)).sum())
        new_360 = int(((ap_days >= 0) & (ap_days <= 360)).sum())
        # 大白话：执行下一行代码：out[\"boss_new_account_ratio_360d\"] = safe_div(float(new_360), float(total_accounts))
        out["boss_new_account_ratio_360d"] = safe_div(float(new_360), float(total_accounts))
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_open_speed_per_month\"] = SENTINEL
        out["boss_open_speed_per_month"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_new_account_ratio_360d\"] = SENTINEL
        out["boss_new_account_ratio_360d"] = SENTINEL

    # -------------------------
    # C) consultas：查询类
    # -------------------------
    # 大白话：执行下一行代码：total_queries = int(len(cst)) if cst is not None else 0
    total_queries = int(len(cst)) if cst is not None else 0

    # 平均查询间隔（天）
    # 大白话：执行下一行代码：if total_queries >= 2:
    if total_queries >= 2:
        # 大白话：执行下一行代码：fc = cst[\"fechaConsulta\"].dropna().sort_values()
        fc = cst["fechaConsulta"].dropna().sort_values()
        # 大白话：执行下一行代码：if len(fc) >= 2:
        if len(fc) >= 2:
            # 大白话：执行下一行代码：diffs = fc.diff().dropna().dt.total_seconds() / 86400.0
            diffs = fc.diff().dropna().dt.total_seconds() / 86400.0
            # 大白话：执行下一行代码：out[\"boss_avg_query_interval_days\"] = float(diffs.mean()) if len(diffs) else SENTINEL
            out["boss_avg_query_interval_days"] = float(diffs.mean()) if len(diffs) else SENTINEL
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_avg_query_interval_days\"] = SENTINEL
            out["boss_avg_query_interval_days"] = SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_avg_query_interval_days\"] = SENTINEL
        out["boss_avg_query_interval_days"] = SENTINEL

    # 距最后查询天数
    # 大白话：执行下一行代码：if total_queries and pd.notna(cutoff):
    if total_queries and pd.notna(cutoff):
        # 大白话：执行下一行代码：fc = cst[\"fechaConsulta\"].dropna()
        fc = cst["fechaConsulta"].dropna()
        # 大白话：执行下一行代码：out[\"boss_days_since_last_query\"] = float(days_between(cutoff, fc.max())) if len(fc) else SENTINEL
        out["boss_days_since_last_query"] = float(days_between(cutoff, fc.max())) if len(fc) else SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_days_since_last_query\"] = SENTINEL
        out["boss_days_since_last_query"] = SENTINEL

    # 查询类型数量（tipoCredito 去重）
    # 大白话：执行下一行代码：out[\"boss_query_tipo_credito_nunique\"] = float(cst[\"tipoCredito\"].replace(\"\", np.nan).dropna().nunique()) if total_queries else 0.0
    out["boss_query_tipo_credito_nunique"] = float(cst["tipoCredito"].replace("", np.nan).dropna().nunique()) if total_queries else 0.0

    # 车贷查询次数 / 信用卡查询次数（全量）
    # 大白话：执行下一行代码：if total_queries:
    if total_queries:
        # 大白话：执行下一行代码：tc = cst[\"tipoCredito\"].fillna(\"\")
        tc = cst["tipoCredito"].fillna("")
        # 大白话：执行下一行代码：out[\"boss_auto_loan_query_cnt\"] = float(int(tc.isin([\"CA\", \"AB\", \"AA\"]).sum()))
        out["boss_auto_loan_query_cnt"] = float(int(tc.isin(["CA", "AB", "AA"]).sum()))
        # 大白话：执行下一行代码：out[\"boss_credit_card_query_cnt\"] = float(int((tc == \"TC\").sum()))
        out["boss_credit_card_query_cnt"] = float(int((tc == "TC").sum()))
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_auto_loan_query_cnt\"] = 0.0
        out["boss_auto_loan_query_cnt"] = 0.0
        # 大白话：执行下一行代码：out[\"boss_credit_card_query_cnt\"] = 0.0
        out["boss_credit_card_query_cnt"] = 0.0

    # 查询频率（次/月）：近360天查询数/12
    # 大白话：执行下一行代码：if total_queries and pd.notna(cutoff):
    if total_queries and pd.notna(cutoff):
        # 大白话：执行下一行代码：in360 = cst[cst[\"fechaConsulta\"].apply(lambda x: in_cutoff_window(cutoff, x, 360))]
        in360 = cst[cst["fechaConsulta"].apply(lambda x: in_cutoff_window(cutoff, x, 360))]
        # 大白话：执行下一行代码：out[\"boss_query_freq_per_month_360d\"] = float(len(in360)) / 12.0
        out["boss_query_freq_per_month_360d"] = float(len(in360)) / 12.0
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_query_freq_per_month_360d\"] = SENTINEL
        out["boss_query_freq_per_month_360d"] = SENTINEL

    # 查询激增标记：近30天查询 > 近90天月均*2
    # 大白话：执行下一行代码：if total_queries and pd.notna(cutoff):
    if total_queries and pd.notna(cutoff):
        # 大白话：执行下一行代码：c30 = len(cst[cst[\"fechaConsulta\"].apply(lambda x: in_cutoff_window(cutoff, x, 30))])
        c30 = len(cst[cst["fechaConsulta"].apply(lambda x: in_cutoff_window(cutoff, x, 30))])
        # 大白话：执行下一行代码：c90 = len(cst[cst[\"fechaConsulta\"].apply(lambda x: in_cutoff_window(cutoff, x, 90))])
        c90 = len(cst[cst["fechaConsulta"].apply(lambda x: in_cutoff_window(cutoff, x, 90))])
        # 大白话：执行下一行代码：monthly_avg_90 = float(c90) / 3.0
        monthly_avg_90 = float(c90) / 3.0
        # 大白话：执行下一行代码：out[\"boss_query_surge_flag\"] = 1.0 if (float(c30) > monthly_avg_90 * 2.0) else 0.0
        out["boss_query_surge_flag"] = 1.0 if (float(c30) > monthly_avg_90 * 2.0) else 0.0
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_query_surge_flag\"] = SENTINEL
        out["boss_query_surge_flag"] = SENTINEL

    # 窗口内查询金额 mean/max/sum（用 fechaConsulta 筛窗口）
    # 大白话：执行下一行代码：if total_queries and pd.notna(cutoff):
    if total_queries and pd.notna(cutoff):
        # 大白话：执行下一行代码：for w in WINDOWS:
        for w in WINDOWS:
            # 大白话：执行下一行代码：dfw = cst[cst[\"fechaConsulta\"].apply(lambda x: in_cutoff_window(cutoff, x, w))]
            dfw = cst[cst["fechaConsulta"].apply(lambda x: in_cutoff_window(cutoff, x, w))]
            # 大白话：执行下一行代码：vals = dfw[\"importeCredito\"].dropna()
            vals = dfw["importeCredito"].dropna()
            # Excel 说 sum/count，且总和只统计 >0
            # 大白话：执行下一行代码：vals_pos = vals[vals > 0]
            vals_pos = vals[vals > 0]
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_sum\"] = float(vals_pos.sum()) if len(vals_pos) else 0.0
            out[f"boss_w{w}d_query_amount_sum"] = float(vals_pos.sum()) if len(vals_pos) else 0.0
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_max\"] = float(vals_pos.max()) if len(vals_pos) else SENTINEL
            out[f"boss_w{w}d_query_amount_max"] = float(vals_pos.max()) if len(vals_pos) else SENTINEL
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_mean\"] = float(vals_pos.sum() / len(vals_pos)) if len(vals_pos) else SENTINEL
            out[f"boss_w{w}d_query_amount_mean"] = float(vals_pos.sum() / len(vals_pos)) if len(vals_pos) else SENTINEL
    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：for w in WINDOWS:
        for w in WINDOWS:
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_sum\"] = 0.0
            out[f"boss_w{w}d_query_amount_sum"] = 0.0
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_max\"] = SENTINEL
            out[f"boss_w{w}d_query_amount_max"] = SENTINEL
            # 大白话：执行下一行代码：out[f\"boss_w{w}d_query_amount_mean\"] = SENTINEL
            out[f"boss_w{w}d_query_amount_mean"] = SENTINEL

    # -------------------------
    # D) empleos：工作类
    # -------------------------
    # 大白话：执行下一行代码：work_cnt = int(len(emp)) if emp is not None else 0
    work_cnt = int(len(emp)) if emp is not None else 0
    # 大白话：执行下一行代码：out[\"boss_work_records_cnt\"] = float(work_cnt)
    out["boss_work_records_cnt"] = float(work_cnt)

    # 大白话：执行下一行代码：if work_cnt:
    if work_cnt:
        # 平均月薪
        # 大白话：执行下一行代码：sm = emp[\"salarioMensual\"].dropna()
        sm = emp["salarioMensual"].dropna()
        # 大白话：执行下一行代码：out[\"boss_avg_salary\"] = float(sm.mean()) if len(sm) else SENTINEL
        out["boss_avg_salary"] = float(sm.mean()) if len(sm) else SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_salary\"] = float(sm.max()) if len(sm) else SENTINEL
        out["boss_max_salary"] = float(sm.max()) if len(sm) else SENTINEL

        # 当前月薪
        # 大白话：执行下一行代码：out[\"boss_current_salary\"] = float(current_salary) if pd.notna(current_salary) else SENTINEL
        out["boss_current_salary"] = float(current_salary) if pd.notna(current_salary) else SENTINEL

        # 有验证工作标记：任意一条有 fechaVerificacionEmpleo
        # 大白话：执行下一行代码：out[\"boss_has_verified_employment_flag\"] = 1.0 if emp[\"fechaVerificacionEmpleo\"].notna().any() else 0.0
        out["boss_has_verified_employment_flag"] = 1.0 if emp["fechaVerificacionEmpleo"].notna().any() else 0.0

        # 薪资已验证标记：有验证且 salarioMensual 有值
        # 大白话：执行下一行代码：out[\"boss_salary_verified_flag\"] = 1.0 if (emp[\"fechaVerificacionEmpleo\"].notna() & emp[\"salarioMensual\"].notna()).any() else 0.0
        out["boss_salary_verified_flag"] = 1.0 if (emp["fechaVerificacionEmpleo"].notna() & emp["salarioMensual"].notna()).any() else 0.0

        # 工作变更次数：公司名称去重数
        # 大白话：执行下一行代码：out[\"boss_company_nunique\"] = float(emp[\"nombreEmpresa\"].replace(\"\", np.nan).dropna().nunique())
        out["boss_company_nunique"] = float(emp["nombreEmpresa"].replace("", np.nan).dropna().nunique())

        # 工作月数：每条工作 start=fechaContratacion，end=fechaUltimoDiaEmpleo 否则 cutoff
        # 大白话：执行下一行代码：months = []
        months = []
        # 大白话：执行下一行代码：for _, rr in emp.iterrows():
        for _, rr in emp.iterrows():
            # 大白话：执行下一行代码：st = rr[\"fechaContratacion\"]
            st = rr["fechaContratacion"]
            # 大白话：执行下一行代码：ed = rr[\"fechaUltimoDiaEmpleo\"] if pd.notna(rr[\"fechaUltimoDiaEmpleo\"]) else cutoff
            ed = rr["fechaUltimoDiaEmpleo"] if pd.notna(rr["fechaUltimoDiaEmpleo"]) else cutoff
            # 大白话：执行下一行代码：m = compute_job_months(st, ed)
            m = compute_job_months(st, ed)
            # 大白话：执行下一行代码：if not np.isnan(m):
            if not np.isnan(m):
                # 大白话：执行下一行代码：months.append(m)
                months.append(m)
        # 大白话：执行下一行代码：out[\"boss_avg_job_months\"] = float(np.mean(months)) if len(months) else SENTINEL
        out["boss_avg_job_months"] = float(np.mean(months)) if len(months) else SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_job_months\"] = float(np.max(months)) if len(months) else SENTINEL
        out["boss_max_job_months"] = float(np.max(months)) if len(months) else SENTINEL

        # 当前工作月数：当前工作（未离职优先）fechaContratacion 到 cutoff
        # 大白话：执行下一行代码：cur_emp = emp[emp[\"fechaUltimoDiaEmpleo\"].isna()]
        cur_emp = emp[emp["fechaUltimoDiaEmpleo"].isna()]
        # 大白话：执行下一行代码：cand = cur_emp if len(cur_emp) else emp
        cand = cur_emp if len(cur_emp) else emp
        # 大白话：执行下一行代码：cand = cand.sort_values(by=[\"fechaContratacion\"], ascending=[False])
        cand = cand.sort_values(by=["fechaContratacion"], ascending=[False])
        # 大白话：执行下一行代码：st = cand.iloc[0][\"fechaContratacion\"]
        st = cand.iloc[0]["fechaContratacion"]
        # 大白话：执行下一行代码：cur_m = compute_job_months(st, cutoff)
        cur_m = compute_job_months(st, cutoff)
        # 大白话：执行下一行代码：out[\"boss_current_job_months\"] = float(cur_m) if not np.isnan(cur_m) else SENTINEL
        out["boss_current_job_months"] = float(cur_m) if not np.isnan(cur_m) else SENTINEL

        # 近期换工作标记：近180天是否有新工作（以 fechaContratacion 判定）
        # 大白话：执行下一行代码：if pd.notna(cutoff):
        if pd.notna(cutoff):
            # 大白话：执行下一行代码：d = (cutoff - emp[\"fechaContratacion\"]).dt.total_seconds() / 86400.0
            d = (cutoff - emp["fechaContratacion"]).dt.total_seconds() / 86400.0
            # 大白话：执行下一行代码：out[\"boss_recent_job_change_flag_180d\"] = 1.0 if ((d >= 0) & (d <= 180)).any() else 0.0
            out["boss_recent_job_change_flag_180d"] = 1.0 if ((d >= 0) & (d <= 180)).any() else 0.0
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_recent_job_change_flag_180d\"] = SENTINEL
            out["boss_recent_job_change_flag_180d"] = SENTINEL

        # 工作稳定性得分（按 Excel）
        # 大白话：执行下一行代码：if len(months) and pd.notna(out[\"boss_current_job_months\"]):
        if len(months) and pd.notna(out["boss_current_job_months"]):
            # 大白话：执行下一行代码：base = 50.0
            base = 50.0
            # 大白话：执行下一行代码：cur_add = min((int(out[\"boss_current_job_months\"] // 12) * 10.0), 30.0)
            cur_add = min((int(out["boss_current_job_months"] // 12) * 10.0), 30.0)
            # 大白话：执行下一行代码：avg_add = min((int((out[\"boss_avg_job_months\"] if out[\"boss_avg_job_months\"] != SENTINEL else 0.0) // 12) * 5.0), 15.0)
            avg_add = min((int((out["boss_avg_job_months"] if out["boss_avg_job_months"] != SENTINEL else 0.0) // 12) * 5.0), 15.0)
            # 大白话：执行下一行代码：chg_pen = min(max(work_cnt - 1, 0) * 5.0, 25.0)
            chg_pen = min(max(work_cnt - 1, 0) * 5.0, 25.0)
            # 大白话：执行下一行代码：score = base + cur_add + avg_add - chg_pen
            score = base + cur_add + avg_add - chg_pen
            # 大白话：执行下一行代码：out[\"boss_job_stability_score\"] = float(np.clip(score, 0.0, 100.0))
            out["boss_job_stability_score"] = float(np.clip(score, 0.0, 100.0))
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_job_stability_score\"] = SENTINEL
            out["boss_job_stability_score"] = SENTINEL

    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_avg_salary\"] = SENTINEL
        out["boss_avg_salary"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_salary\"] = SENTINEL
        out["boss_max_salary"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_current_salary\"] = SENTINEL
        out["boss_current_salary"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_has_verified_employment_flag\"] = 0.0
        out["boss_has_verified_employment_flag"] = 0.0
        # 大白话：执行下一行代码：out[\"boss_salary_verified_flag\"] = 0.0
        out["boss_salary_verified_flag"] = 0.0
        # 大白话：执行下一行代码：out[\"boss_company_nunique\"] = 0.0
        out["boss_company_nunique"] = 0.0
        # 大白话：执行下一行代码：out[\"boss_avg_job_months\"] = SENTINEL
        out["boss_avg_job_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_job_months\"] = SENTINEL
        out["boss_max_job_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_current_job_months\"] = SENTINEL
        out["boss_current_job_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_recent_job_change_flag_180d\"] = SENTINEL
        out["boss_recent_job_change_flag_180d"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_job_stability_score\"] = SENTINEL
        out["boss_job_stability_score"] = SENTINEL

    # -------------------------
    # E) domicilios：住址类
    # -------------------------
    # 大白话：执行下一行代码：addr_cnt = int(len(dom)) if dom is not None else 0
    addr_cnt = int(len(dom)) if dom is not None else 0
    # 大白话：执行下一行代码：out[\"boss_state_nunique\"] = float(dom[\"estado\"].replace(\"\", np.nan).dropna().nunique()) if addr_cnt else 0.0
    out["boss_state_nunique"] = float(dom["estado"].replace("", np.nan).dropna().nunique()) if addr_cnt else 0.0

    # 大白话：执行下一行代码：if addr_cnt and pd.notna(cutoff):
    if addr_cnt and pd.notna(cutoff):
        # 居住月数（每条地址：cutoff - fechaResidencia）
        # 大白话：执行下一行代码：res_months = []
        res_months = []
        # 大白话：执行下一行代码：for _, rr in dom.iterrows():
        for _, rr in dom.iterrows():
            # 大白话：执行下一行代码：m = compute_res_months(rr[\"fechaResidencia\"], cutoff)
            m = compute_res_months(rr["fechaResidencia"], cutoff)
            # 大白话：执行下一行代码：if not np.isnan(m):
            if not np.isnan(m):
                # 大白话：执行下一行代码：res_months.append(m)
                res_months.append(m)

        # 大白话：执行下一行代码：out[\"boss_avg_res_months\"] = float(np.mean(res_months)) if len(res_months) else SENTINEL
        out["boss_avg_res_months"] = float(np.mean(res_months)) if len(res_months) else SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_res_months\"] = float(np.max(res_months)) if len(res_months) else SENTINEL
        out["boss_max_res_months"] = float(np.max(res_months)) if len(res_months) else SENTINEL

        # 当前居住月数：最新 fechaResidencia
        # 大白话：执行下一行代码：fr = dom[\"fechaResidencia\"].dropna()
        fr = dom["fechaResidencia"].dropna()
        # 大白话：执行下一行代码：if len(fr):
        if len(fr):
            # 大白话：执行下一行代码：cur_start = fr.max()
            cur_start = fr.max()
            # 大白话：执行下一行代码：cur_res_m = compute_res_months(cur_start, cutoff)
            cur_res_m = compute_res_months(cur_start, cutoff)
            # 大白话：执行下一行代码：out[\"boss_current_res_months\"] = float(cur_res_m) if not np.isnan(cur_res_m) else SENTINEL
            out["boss_current_res_months"] = float(cur_res_m) if not np.isnan(cur_res_m) else SENTINEL
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_current_res_months\"] = SENTINEL
            out["boss_current_res_months"] = SENTINEL

        # 近期搬迁标记：最新地址在近180天且与历史地址方向不一致
        # 大白话：执行下一行代码：if len(fr):
        if len(fr):
            # 大白话：执行下一行代码：latest_idx = dom[\"fechaResidencia\"].idxmax()
            latest_idx = dom["fechaResidencia"].idxmax()
            # 大白话：执行下一行代码：latest_addr = str(dom.loc[latest_idx, \"direccion\"]) if latest_idx in dom.index else \"\"
            latest_addr = str(dom.loc[latest_idx, "direccion"]) if latest_idx in dom.index else ""
            # 大白话：执行下一行代码：latest_days = days_between(cutoff, dom.loc[latest_idx, \"fechaResidencia\"]) if latest_idx in dom.index else np.nan
            latest_days = days_between(cutoff, dom.loc[latest_idx, "fechaResidencia"]) if latest_idx in dom.index else np.nan
            # 大白话：执行下一行代码：if (not np.isnan(latest_days)) and (latest_days <= 180):
            if (not np.isnan(latest_days)) and (latest_days <= 180):
                # 大白话：执行下一行代码：other_addrs = dom[\"direccion\"].astype(str)
                other_addrs = dom["direccion"].astype(str)
                # 大白话：执行下一行代码：out[\"boss_recent_move_flag_180d\"] = 1.0 if (other_addrs != latest_addr).any() else 0.0
                out["boss_recent_move_flag_180d"] = 1.0 if (other_addrs != latest_addr).any() else 0.0
            # 大白话：执行下一行代码：else:
            else:
                # 大白话：执行下一行代码：out[\"boss_recent_move_flag_180d\"] = 0.0
                out["boss_recent_move_flag_180d"] = 0.0
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_recent_move_flag_180d\"] = SENTINEL
            out["boss_recent_move_flag_180d"] = SENTINEL

        # 居住稳定性得分（按 Excel）
        # 大白话：执行下一行代码：if (out[\"boss_current_res_months\"] != SENTINEL) and (out[\"boss_avg_res_months\"] != SENTINEL):
        if (out["boss_current_res_months"] != SENTINEL) and (out["boss_avg_res_months"] != SENTINEL):
            # 大白话：执行下一行代码：base = 50.0
            base = 50.0
            # 大白话：执行下一行代码：cur_add = min(int(out[\"boss_current_res_months\"] // 12) * 10.0, 30.0)
            cur_add = min(int(out["boss_current_res_months"] // 12) * 10.0, 30.0)
            # 大白话：执行下一行代码：avg_add = min(int(out[\"boss_avg_res_months\"] // 12) * 5.0, 15.0)
            avg_add = min(int(out["boss_avg_res_months"] // 12) * 5.0, 15.0)
            # 大白话：执行下一行代码：move_pen = min(max(addr_cnt - 1, 0) * 5.0, 20.0)
            move_pen = min(max(addr_cnt - 1, 0) * 5.0, 20.0)
            # 大白话：执行下一行代码：st_cnt = int(dom[\"estado\"].replace(\"\", np.nan).dropna().nunique())
            st_cnt = int(dom["estado"].replace("", np.nan).dropna().nunique())
            # 大白话：执行下一行代码：state_pen = min(max(st_cnt - 1, 0) * 3.0, 15.0)
            state_pen = min(max(st_cnt - 1, 0) * 3.0, 15.0)
            # 大白话：执行下一行代码：score = base + cur_add + avg_add - move_pen - state_pen
            score = base + cur_add + avg_add - move_pen - state_pen
            # 大白话：执行下一行代码：out[\"boss_res_stability_score\"] = float(np.clip(score, 0.0, 100.0))
            out["boss_res_stability_score"] = float(np.clip(score, 0.0, 100.0))
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[\"boss_res_stability_score\"] = SENTINEL
            out["boss_res_stability_score"] = SENTINEL

    # 大白话：执行下一行代码：else:
    else:
        # 大白话：执行下一行代码：out[\"boss_avg_res_months\"] = SENTINEL
        out["boss_avg_res_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_max_res_months\"] = SENTINEL
        out["boss_max_res_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_current_res_months\"] = SENTINEL
        out["boss_current_res_months"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_recent_move_flag_180d\"] = SENTINEL
        out["boss_recent_move_flag_180d"] = SENTINEL
        # 大白话：执行下一行代码：out[\"boss_res_stability_score\"] = SENTINEL
        out["boss_res_stability_score"] = SENTINEL

    # -------------------------
    # F) 统一：保留 2 位小数
    # -------------------------
    # 大白话：执行下一行代码：for k, v in list(out.items()):
    for k, v in list(out.items()):
        # 大白话：执行下一行代码：if v is None:
        if v is None:
            # 大白话：执行下一行代码：out[k] = SENTINEL
            out[k] = SENTINEL
            # 大白话：执行下一行代码：continue
            continue
        # 大白话：执行下一行代码：try:
        try:
            # 大白话：执行下一行代码：fv = float(v)
            fv = float(v)
        # 大白话：执行下一行代码：except Exception:
        except Exception:
            # 大白话：执行下一行代码：out[k] = SENTINEL
            out[k] = SENTINEL
            # 大白话：执行下一行代码：continue
            continue
        # 计数类也统一转 float；保留 2 位
        # 大白话：执行下一行代码：if fv != SENTINEL and not np.isnan(fv):
        if fv != SENTINEL and not np.isnan(fv):
            # 大白话：执行下一行代码：out[k] = float(np.round(fv, 2))
            out[k] = float(np.round(fv, 2))
        # 大白话：执行下一行代码：else:
        else:
            # 大白话：执行下一行代码：out[k] = float(SENTINEL)
            out[k] = float(SENTINEL)

    # -------------------------
    # G) 特征命名：统一加前后缀
    # -------------------------
    # 大白话：你要求所有衍生特征列名都带 `cdc_` 前缀、并以 `_607` 作为后缀。
    #         这里在函数返回前一次性改名，避免到处手工拼字符串。
    # 大白话：执行下一行代码：out = {f\"cdc_{k}_607\": v for k, v in out.items()}
    out = {f"cdc_{k}_607": v for k, v in out.items()}

    # 大白话：执行下一行代码：return out
    return out


In [ ]:
"""大白话：这个代码格负责“批量跑衍生 + 得到最终特征表”。

- 我们会按 apply_id 循环，取该样本自己的 cutoff=request_time。
- 分别拿到该 apply_id 的 consultas/creditos/empleos/domicilios 明细子表。
- 调用上一格的 `derive_one_apply(...)` 得到一行特征 dict。
- 拼成 `features_df`。

重要：不写任何文件，只做 DataFrame 输出与简单展示。
"""

# =========================
# 1) 预分组，加速按 apply_id 取子表
# =========================
# 大白话：执行下一行代码：consultas_g = consultas_df.groupby(\"apply_id\") if len(consultas_df) else {}
consultas_g = consultas_df.groupby("apply_id") if len(consultas_df) else {}
# 大白话：执行下一行代码：creditos_g = creditos_df.groupby(\"apply_id\") if len(creditos_df) else {}
creditos_g = creditos_df.groupby("apply_id") if len(creditos_df) else {}
# 大白话：执行下一行代码：empleos_g = empleos_df.groupby(\"apply_id\") if len(empleos_df) else {}
empleos_g = empleos_df.groupby("apply_id") if len(empleos_df) else {}
# 大白话：执行下一行代码：domicilios_g = domicilios_df.groupby(\"apply_id\") if len(domicilios_df) else {}
domicilios_g = domicilios_df.groupby("apply_id") if len(domicilios_df) else {}

# 大白话：执行下一行代码：rows = []
rows = []

# 大白话：执行下一行代码：for _, r in raw_df[[\"apply_id\", \"request_time\"]].iterrows():
for _, r in raw_df[["apply_id", "request_time"]].iterrows():
    # 大白话：执行下一行代码：apply_id = r[\"apply_id\"]
    apply_id = r["apply_id"]
    # 大白话：执行下一行代码：cutoff = r[\"request_time\"]
    cutoff = r["request_time"]

    # 大白话：执行下一行代码：cst = consultas_g.get_group(apply_id) if hasattr(consultas_g, \"get_group\") and apply_id in consultas_g.groups else consultas_df.iloc[0:0]
    cst = consultas_g.get_group(apply_id) if hasattr(consultas_g, "get_group") and apply_id in consultas_g.groups else consultas_df.iloc[0:0]
    # 大白话：执行下一行代码：cre = creditos_g.get_group(apply_id) if hasattr(creditos_g, \"get_group\") and apply_id in creditos_g.groups else creditos_df.iloc[0:0]
    cre = creditos_g.get_group(apply_id) if hasattr(creditos_g, "get_group") and apply_id in creditos_g.groups else creditos_df.iloc[0:0]
    # 大白话：执行下一行代码：emp = empleos_g.get_group(apply_id) if hasattr(empleos_g, \"get_group\") and apply_id in empleos_g.groups else empleos_df.iloc[0:0]
    emp = empleos_g.get_group(apply_id) if hasattr(empleos_g, "get_group") and apply_id in empleos_g.groups else empleos_df.iloc[0:0]
    # 大白话：执行下一行代码：dom = domicilios_g.get_group(apply_id) if hasattr(domicilios_g, \"get_group\") and apply_id in domicilios_g.groups else domicilios_df.iloc[0:0]
    dom = domicilios_g.get_group(apply_id) if hasattr(domicilios_g, "get_group") and apply_id in domicilios_g.groups else domicilios_df.iloc[0:0]

    # 大白话：执行下一行代码：feat = derive_one_apply(apply_id, cutoff, cst, cre, emp, dom)
    feat = derive_one_apply(apply_id, cutoff, cst, cre, emp, dom)
    # 大白话：执行下一行代码：feat[\"apply_id\"] = apply_id
    feat["apply_id"] = apply_id
    # 大白话：执行下一行代码：feat[\"request_time\"] = cutoff
    feat["request_time"] = cutoff
    # 大白话：执行下一行代码：rows.append(feat)
    rows.append(feat)

# 大白话：执行下一行代码：features_df = pd.DataFrame(rows)
features_df = pd.DataFrame(rows)

# 大白话：把 apply_id 和 request_time 放在最前面，便于查看。
# 大白话：执行下一行代码：cols = [\"apply_id\", \"request_time\"] + [c for c in features_df.columns if c not in [\"apply_id\", \"request_time\"]]
cols = ["apply_id", "request_time"] + [c for c in features_df.columns if c not in ["apply_id", "request_time"]]
# 大白话：执行下一行代码：features_df = features_df.loc[:, cols]
features_df = features_df.loc[:, cols]

# 大白话：执行下一行代码：print(\"features_df shape:\", features_df.shape)
print("features_df shape:", features_df.shape)
# 大白话：执行下一行代码：print(\"features_df head:\")
print("features_df head:")
# 大白话：执行下一行代码：display(features_df.head(5))
display(features_df.head(5))

# 大白话：如果你希望核对“与 Excel 条目数量是否一致”，可以看：
# - excel_feature_cnt：Excel 中条目数（如果能读取）
# - features_df.shape[1]-1：特征列数（不含 apply_id）
# 大白话：执行下一行代码：if excel_feature_cnt is not None:
if excel_feature_cnt is not None:
    # 大白话：执行下一行代码：print(\"excel_feature_cnt:\", excel_feature_cnt)
    print("excel_feature_cnt:", excel_feature_cnt)
# 大白话：执行下一行代码：print(\"feature_col_cnt (exclude apply_id):\", features_df.shape[1] - 1)
print("feature_col_cnt (exclude apply_id):", features_df.shape[1] - 1)

# ===================== 特征字典（3列）=====================
# 大白话：你这次的要求是——特征字典里如果出现了“原始字段名”，就必须把字段含义解释清楚，别人拿到字典才看得懂。
# 大白话：BOSS 板块的特征都在 derive_one_apply 里逐条计算，因此这里用“解析 notebook 源码”的方式：
# - 从 derive_one_apply 的 out["boss_xxx"] = ... 语句里提取表达式
# - 识别表达式中引用到的原始字段名（如 cre["fechaCierreCuenta"]）
# - 把这些字段名拼成“字段名=中文含义”写入 logic_desc

WRITE_FEATURE_DICT_OUTPUTS = True

# 大白话：BOSS 板块用到的原始字段释义（出现字段名就用它解释）。
RAW_FIELD_DESC_BOSS = {
    # consultas
    "fechaConsulta": "查询日期",
    "importeCredito": "授信/贷款金额（合同金额）",
    "tipoCredito": "信用/贷款类型",
    # creditos
    "fechaReporte": "上报日期（窗口锚点）",
    "fechaAperturaCuenta": "账户开立日期（开户日期）",
    "fechaCierreCuenta": "账户关闭日期",
    "tipoResponsabilidad": "责任类型（I个人/M联名/A担保/授权等）",
    "saldoActual": "当前总待还金额",
    "limiteCredito": "信用额度",
    "frecuenciaPagos": "还款频率",
    "tipoCuenta": "账户类型",
    "fechaPeorAtraso": "最严重逾期日期",
    "fechaUltimoPago": "最后还款日期",
    "fechaUltimaCompra": "最后消费日期",
    "fechaActualizacion": "更新日期",
    "ultimaFechaSaldoCero": "最近一次余额为零日期",
    "clavePrevencion": "预警/风险提示代码",
    "claveUnidadMonetaria": "货币单位代码",
    "numeroPagos": "还款期数/还款次数",
    "peorAtraso": "最严重逾期（最大拖欠程度/期数）",
    "registroImpugnado": "异议/争议记录标志（是否被申诉）",
    "totalPagosReportados": "已上报还款次数（总计）",
    "saldoVencido": "逾期余额",
    # empleos
    "salarioMensual": "月薪",
    "fechaContratacion": "雇佣日期",
    "fechaUltimoDiaEmpleo": "最后工作日",
    "fechaVerificacionEmpleo": "工作信息验证日期",
    "nombreEmpresa": "公司名称",
    # domicilios
    "fechaResidencia": "居住日期（开始居住日期）",
    "estado": "州/省（缩写）",
    "direccion": "街道地址",
    # 通用
    "request_time": "截止日期（本次申请时间）",
}

# 大白话：从 notebook 源码里解析 derive_one_apply 的 out["boss_xxx"] 赋值语句。
import re
import json
from pathlib import Path

_nb = json.loads(Path("BOSS板块衍生.ipynb").read_text(encoding="utf-8"))
_cell2_src = "".join(_nb["cells"][2]["source"])  # cell2: helper + derive_one_apply

# 抓取每条 out["boss_xxx"] = ... 的表达式
# 大白话：注意这里解析的是 cell2 里的“真实 python 代码行”，形如：out["boss_xxx"] = ...
_assign_pat = re.compile(r'^\s*out\["(boss_[A-Za-z0-9_]+)"\]\s*=\s*(.+?)\s*$')

# 大白话：除了 out[...] 这一行本身，很多特征会先用中间变量（如 closed_cnt）承接字段计算。
# 所以这里不仅保存 out[...] 的右侧表达式，还会取“out[...] 上方若干行代码”一起扫描，尽量抓到 cre["xxx"] 这类原始字段引用。
_cell2_lines = _cell2_src.splitlines()

_feat_expr_map = {}
_feat_ctx_map = {}
for _i, _line in enumerate(_cell2_lines):
    _m = _assign_pat.match(_line)
    if not _m:
        continue
    _k = _m.group(1)  # boss_xxx
    _feat_expr_map[_k] = _m.group(2)
    # 往上回看 12 行（经验值）：尽量覆盖“中间变量=cre[字段]...”的定义，同时减少把无关字段也扫进来的概率
    _ctx = "\n".join(_cell2_lines[max(0, _i - 12) : _i + 1])
    _feat_ctx_map[_k] = _ctx

# 大白话：从源码片段里找出引用的原始字段名（形如 cre["xxx"] / cst["xxx"] / emp["xxx"] / dom["xxx"]）。
# 说明：我们要匹配的是“方括号+双引号”的真实文本片段：["campo"]
_field_pat = re.compile(r'\["([A-Za-z0-9_]+)"\]')

def _extract_raw_fields(text: str):
    found = []
    for f in _field_pat.findall(text or ""):
        if f in RAW_FIELD_DESC_BOSS and f not in found:
            found.append(f)
    return found

# 大白话：把 features_df 的列名（cdc_boss_xxx_607）映射回 boss_xxx。
_boss_cols = [c for c in features_df.columns if c != "apply_id"]

def _to_boss_key(col: str) -> str:
    # 期望格式：cdc_boss_xxx_607
    if col.startswith("cdc_") and col.endswith("_607"):
        return col[len("cdc_") : -len("_607")]
    return col

# 大白话：生成 3 列特征字典：feature_name / cn_desc / logic_desc
_rows = []
for _col in _boss_cols:
    _boss_key = _to_boss_key(_col)  # boss_xxx
    _expr = _feat_expr_map.get(_boss_key, "")
    # 大白话：优先用“上下文源码片段”去抓原始字段（更容易捕捉到中间变量的来源字段）。
    _ctx_text = _feat_ctx_map.get(_boss_key, _expr)
    _raw_fields = _extract_raw_fields(_ctx_text)

    # 大白话：logic_desc 里只要出现字段名，就必须解释字段含义。
    if _raw_fields:
        _raw_explain = "；".join([f"{f}={RAW_FIELD_DESC_BOSS.get(f, '')}" for f in _raw_fields])
        _logic_desc = f"expr={_expr}; raw_fields[{_raw_explain}]"
    else:
        _logic_desc = f"expr={_expr}; raw_fields[未在表达式中识别到具体原始字段名（可能是常量/中间变量/函数封装）]"

    _rows.append(
        {
            "feature_name": _col,
            # 大白话：cn_desc 兜底先写成“BOSS板块特征：{boss_key}”。
            # 如果你后续希望把 Excel 的“中文释义”逐条严格对齐到每个 feature_name，需要再提供“Excel行 -> feature_name”的对应关系。
            "cn_desc": f"BOSS板块特征：{_boss_key}",
            "logic_desc": _logic_desc,
        }
    )

boss_feature_dict_3col = pd.DataFrame(_rows)
print("boss_feature_dict_3col shape:", boss_feature_dict_3col.shape)
display(boss_feature_dict_3col.head(10))

# 大白话：默认不写出文件；你确认后再打开。
if WRITE_FEATURE_DICT_OUTPUTS:
    OUT_PATH = Path("outputs") / "feature_dict_boss_3col.csv"
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    boss_feature_dict_3col.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print("[WRITE] boss feature dict written:", str(OUT_PATH.resolve()))
else:
    print("[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出 BOSS 特征字典（等你确认后再输出）")



features_df shape: (12546, 149)
features_df head:


,apply_id,cdc_boss_total_accounts_607,cdc_boss_closed_accounts_cnt_607,cdc_boss_open_accounts_cnt_607,cdc_boss_zero_balance_accounts_cnt_607,cdc_boss_respons_individual_cnt_607,cdc_boss_respons_joint_cnt_607,cdc_boss_respons_guarantor_a_cnt_607,cdc_boss_individual_balance_sum_607,cdc_boss_joint_balance_sum_607,...,cdc_boss_max_job_months_607,cdc_boss_current_job_months_607,cdc_boss_recent_job_change_flag_180d_607,cdc_boss_job_stability_score_607,cdc_boss_state_nunique_607,cdc_boss_avg_res_months_607,cdc_boss_max_res_months_607,cdc_boss_current_res_months_607,cdc_boss_recent_move_flag_180d_607,cdc_boss_res_stability_score_607
0,1065991091661283329,142.0,111.0,31.0,115.0,142.0,0.0,0.0,108820.0,0.0,...,-999.0,-999.0,-999.0,-999.0,1.0,11.38,24.90,0.00,1.0,40.0
1,1066560157648134145,16.0,6.0,10.0,8.0,16.0,0.0,0.0,1288276.0,0.0,...,-999.0,-999.0,-999.0,-999.0,1.0,24.12,72.28,0.01,1.0,50.0
2,1066719243236777985,14.0,9.0,5.0,10.0,13.0,0.0,0.0,25156.0,0.0,...,-999.0,-999.0,-999.0,-999.0,2.0,24.37,50.61,0.01,1.0,47.0
3,1067025354145890305,73.0,53.0,20.0,56.0,73.0,0.0,0.0,663031.0,0.0,...,-999.0,-999.0,-999.0,-999.0,1.0,1.96,4.05,0.01,1.0,40.0
4,1067575625355862017,20.0,14.0,6.0,15.0,15.0,0.0,0.0,36698.0,0.0,...,-999.0,-999.0,0.0,-999.0,1.0,18.13,40.59,0.02,1.0,45.0


feature_col_cnt (exclude apply_id): 148
boss_feature_dict_3col shape: (148, 3)


,feature_name,cn_desc,logic_desc
0,cdc_boss_total_accounts_607,BOSS板块特征：boss_total_accounts,expr=float(total_accounts); raw_fields[未在表达式中识...
1,cdc_boss_closed_accounts_cnt_607,BOSS板块特征：boss_closed_accounts_cnt,expr=float(closed_cnt); raw_fields[fechaCierre...
2,cdc_boss_open_accounts_cnt_607,BOSS板块特征：boss_open_accounts_cnt,expr=float(open_cnt); raw_fields[fechaCierreCu...
3,cdc_boss_zero_balance_accounts_cnt_607,BOSS板块特征：boss_zero_balance_accounts_cnt,expr=float(zero_balance_cnt); raw_fields[fecha...
4,cdc_boss_respons_individual_cnt_607,BOSS板块特征：boss_respons_individual_cnt,"expr=float(int((tipo == ""I"").sum())) if total_..."
5,cdc_boss_respons_joint_cnt_607,BOSS板块特征：boss_respons_joint_cnt,"expr=float(int((tipo == ""M"").sum())) if total_..."
6,cdc_boss_respons_guarantor_a_cnt_607,BOSS板块特征：boss_respons_guarantor_a_cnt,"expr=float(int((tipo == ""A"").sum())) if total_..."
7,cdc_boss_individual_balance_sum_607,BOSS板块特征：boss_individual_balance_sum,"expr=float(saldo[tipo == ""I""].sum()) if total_..."
8,cdc_boss_joint_balance_sum_607,BOSS板块特征：boss_joint_balance_sum,"expr=float(saldo[tipo == ""M""].sum()) if total_..."
9,cdc_boss_tipo_cuenta_nunique_607,BOSS板块特征：boss_tipo_cuenta_nunique,"expr=float(cre[""tipoCuenta""].replace("""", np.na..."


[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出 BOSS 特征字典（等你确认后再输出）


In [9]:
# ===================== 输出完整 features_df 到 CSV =====================
# 大白话：将 features_df 全量输出到 CSV 文件
WRITE_FEATURES_CSV = True  # 设置为 True 输出 CSV，False 则跳过
WRITE_SAMPLE_200 = True  # 设置为 True 输出前200条样本，False 则跳过

if WRITE_FEATURES_CSV:
    from datetime import datetime
    
    # 生成带时间戳的文件名
    # timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # csv_filename = f"boss_features_full_{timestamp}.csv"
    csv_filename = f"cdcboss_features_full_data.csv"

    csv_path = Path("outputs") / csv_filename
    
    # 确保 outputs 目录存在
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    
    # 输出完整的 features_df 到 CSV
    features_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    
    # 输出前200条记录的小文件（用于快速查看和测试）
    if WRITE_SAMPLE_200:
        csv_sample200_filename = "cdcboss_features_sample200.csv"
        csv_sample200_path = Path("outputs") / csv_sample200_filename
        features_sample200 = features_df.head(200)  # 取前200条记录
        features_sample200.to_csv(csv_sample200_path, index=False, encoding="utf-8-sig")
        print(f"[WRITE] BOSS 特征前200条样本已输出到: {csv_sample200_path.resolve()}")
        print(f"[INFO] 样本数据形状: {features_sample200.shape}")
    
    
    print(f"[WRITE] BOSS 特征全量数据已输出到: {csv_path.resolve()}")
    print(f"[INFO] 输出数据形状: {features_df.shape}")
    print(f"[INFO] 包含 {features_df.shape[0]} 行数据，{features_df.shape[1]} 列特征（含 apply_id）")
else:
    print("[INFO] WRITE_FEATURES_CSV=False，跳过 CSV 输出")

[WRITE] BOSS 特征全量数据已输出到: /Users/zhanglifeng12703/Documents/OverseasPython/CDC/outputs/cdcboss_features_full_data.csv
[INFO] 输出数据形状: (12546, 149)
[INFO] 包含 12546 行数据，149 列特征（含 apply_id）


In [10]:
# TODO（衍生报告：把评估结果写进一个 Excel，多 sheet）
# 大白话：你确认了“评估结果要写进一个 Excel 作为衍生报告”，所以这一格会把 BOSS 板块的评估输出落盘。
# 说明：
# - BOSS 板块默认原始表变量名是 raw_df，特征表变量名是 features_df。
# - 如果 raw_df 里有 fpd7/approve_state：会额外产出分周坏率、IV、PSI、TopIV 分箱、分周IV。
# - 如果没有标签列：只输出特征画像（空值率/同一值率）与元信息。

import re  # 用于清洗 sheet 名
from pathlib import Path  # 用于拼接输出路径

import numpy as np  # 数值计算
import pandas as pd  # 表格计算

# 大白话：写报告开关（你已明确要写，所以默认 True）。
WRITE_REPORTS = False

# 大白话：输出文件路径（BOSS 板块报告）。
REPORT_XLSX = Path("outputs") / "boss_eval_report.xlsx"

# 大白话：统一把 NaN 或 -999 当作空值。
SENTINEL_VALUE = -999

# 大白话：Excel 的 sheet 名最长 31 且不能重复，这里做一个安全命名。
def _safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))[:31] or "sheet"
    s = base
    i = 1
    while s in used:
        suf = f"_{i}"
        s = base[: max(0, 31 - len(suf))] + suf
        i += 1
    used.add(s)
    return s

# 大白话：拿到原始数据与特征表。
_raw = raw_df if "raw_df" in globals() else None
_feat = features_df if "features_df" in globals() else None

if _raw is None or _feat is None:
    raise RuntimeError("[REPORT] 未找到 raw_df 或 features_df，请先从头运行 notebook 生成它们。")

# 大白话：BOSS 的 features_df 里 apply_id 是普通列，这里先设为索引便于对齐。
_feat2 = _feat.copy()
if "apply_id" in _feat2.columns:
    _feat2 = _feat2.set_index("apply_id")

# 大白话：request_time 用 raw_df 的 request_time（BOSS 特征表里通常没有 request_time 列）。
if "request_time" in _raw.columns:
    req_time = (
        _raw[["apply_id", "request_time"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["request_time"]
        .pipe(pd.to_datetime, errors="coerce")
        .reindex(_feat2.index)
    )
else:
    req_time = pd.Series(pd.NaT, index=_feat2.index)

# 大白话：X 是所有衍生特征列。
X = _feat2.copy()

# 大白话：空值掩码（NaN 或 -999）。
missing_mask = X.isna() | (X == SENTINEL_VALUE)

# 大白话：空值率。
na_rate = missing_mask.mean(axis=0)

# 大白话：同一值率。
_same = {}
for c in X.columns:
    s = X[c].copy()
    s = s.mask(missing_mask[c], "__MISSING__")
    vc = s.value_counts(dropna=False, normalize=True)
    _same[c] = float(vc.iloc[0]) if len(vc) > 0 else 1.0
same_value_rate = pd.Series(_same).reindex(X.columns)

# 大白话：标签与 approve_state（如果存在）。
has_fpd7 = "fpd7" in _raw.columns
has_state = "approve_state" in _raw.columns

if has_fpd7:
    y = (
        _raw[["apply_id", "fpd7"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["fpd7"]
        .pipe(pd.to_numeric, errors="coerce")
        .reindex(_feat2.index)
    )
    y01 = (y > 0).astype("int8")
else:
    y = None
    y01 = None

if has_state:
    state = (
        _raw[["apply_id", "approve_state"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["approve_state"]
        .fillna("")
        .astype(str)
        .str.lower()
        .reindex(_feat2.index)
        .fillna("")
    )
    mask_cycle = state.str.contains("cycle_pass", na=False)
    mask_single = state.str.contains("single_pass", na=False)
else:
    mask_cycle = pd.Series(False, index=_feat2.index)
    mask_single = pd.Series(False, index=_feat2.index)

mask_all = pd.Series(True, index=_feat2.index)

# 大白话：分周（周一 00:00:00）。
week_start = (req_time - pd.to_timedelta(req_time.dt.weekday, unit="D")).dt.normalize()

# ========== 可写入 Excel 的表集合 ==========
TABLES: dict[str, pd.DataFrame] = {}

# 1) 特征画像
profile_df = (
    pd.DataFrame({"feature": X.columns, "na_rate": na_rate.values, "same_value_rate": same_value_rate.values})
    .sort_values(["na_rate", "same_value_rate"], ascending=[False, False])
    .reset_index(drop=True)
)
TABLES["feature_profile"] = profile_df

# 2) 分周坏率（需要 y01）
if y01 is not None:
    def _weekly_bad_rate(mask: pd.Series, name: str) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out = tmp.groupby("week", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["sample"] = name
        return out

    weekly_bad = pd.concat(
        [
            _weekly_bad_rate(mask_all, "overall"),
            _weekly_bad_rate(mask_cycle, "cycle_pass"),
            _weekly_bad_rate(mask_single, "single_pass"),
        ],
        axis=0,
        ignore_index=True,
    ).sort_values(["week", "sample"]).reset_index(drop=True)

    TABLES["weekly_bad_rate"] = weekly_bad

# 3) IV / PSI / TopIV 分箱 / 分周 IV（需要 y01）
if y01 is not None:
    IV_Q = 10
    BIN_N = 5

    def _compute_iv_one(x: pd.Series, ybin: pd.Series) -> float:
        dfv = pd.DataFrame({"x": x, "y": ybin}).dropna(subset=["y"])
        if dfv.shape[0] == 0:
            return float("nan")
        x_raw = dfv["x"]
        yb = dfv["y"].astype(int)
        miss = x_raw.isna() | (x_raw == SENTINEL_VALUE)
        x_non = x_raw[~miss]
        if x_non.nunique(dropna=True) <= 2:
            b = x_raw.astype(object).where(~miss, "MISSING")
        else:
            try:
                b_non = pd.qcut(x_non, q=IV_Q, duplicates="drop")
                b = pd.Series("MISSING", index=x_raw.index, dtype="object")
                b.loc[~miss] = b_non.astype(str)
            except Exception:
                b = x_raw.astype(object).where(~miss, "MISSING")
        grp = pd.DataFrame({"b": b, "y": yb}).groupby("b", observed=False)["y"].agg(["count", "sum"]).rename(columns={"sum": "bad"})
        grp["good"] = grp["count"] - grp["bad"]
        bt = grp["bad"].sum(); gt = grp["good"].sum()
        if bt == 0 or gt == 0:
            return 0.0
        k = grp.shape[0]
        grp["bad_dist"] = (grp["bad"] + 0.5) / (bt + 0.5 * k)
        grp["good_dist"] = (grp["good"] + 0.5) / (gt + 0.5 * k)
        woe = np.log(grp["bad_dist"] / grp["good_dist"])
        iv = ((grp["bad_dist"] - grp["good_dist"]) * woe).sum()
        return float(iv)

    eligible = (na_rate < 0.9) & (same_value_rate < 0.99)
    cols = X.columns[eligible].tolist()

    iv_overall = {c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce"), y01) for c in cols}
    iv_overall_s = pd.Series(iv_overall, name="iv_overall")

    iv_cycle_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_cycle], y01[mask_cycle]) for c in cols}, name="iv_cycle_pass") if has_state and int(mask_cycle.sum())>0 else pd.Series(dtype="float64", name="iv_cycle_pass")
    iv_single_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_single], y01[mask_single]) for c in cols}, name="iv_single_pass") if has_state and int(mask_single.sum())>0 else pd.Series(dtype="float64", name="iv_single_pass")

    weeks = sorted([w for w in week_start.dropna().unique()])
    first_w = weeks[0] if len(weeks) >= 1 else None
    last_two = weeks[-2:] if len(weeks) >= 2 else weeks
    base_mask = (week_start == first_w) if first_w is not None else pd.Series(False, index=_feat2.index)
    comp_mask = week_start.isin(last_two) if len(last_two) > 0 else pd.Series(False, index=_feat2.index)

    PSI_Q = 10
    EPS = 1e-6

    def _psi_one(x: pd.Series) -> float:
        miss = x.isna() | (x == SENTINEL_VALUE)
        xb = x[base_mask]; xc = x[comp_mask]
        mb = miss[base_mask]; mc = miss[comp_mask]
        if xb.shape[0] == 0 or xc.shape[0] == 0:
            return float("nan")
        xb_non = xb[~mb]
        if xb_non.nunique(dropna=True) <= 2:
            bb = xb.astype(object).where(~mb, "MISSING")
            bc = xc.astype(object).where(~mc, "MISSING")
        else:
            try:
                _, edges = pd.qcut(xb_non, q=PSI_Q, retbins=True, duplicates="drop")
                edges = sorted(set(edges.tolist()))
                if len(edges) < 3:
                    bb = xb.astype(object).where(~mb, "MISSING")
                    bc = xc.astype(object).where(~mc, "MISSING")
                else:
                    bb_non = pd.cut(xb_non, bins=edges, include_lowest=True)
                    bc_non = pd.cut(xc[~mc], bins=edges, include_lowest=True)
                    bb = pd.Series("MISSING", index=xb.index, dtype="object"); bc = pd.Series("MISSING", index=xc.index, dtype="object")
                    bb.loc[~mb] = bb_non.astype(str); bc.loc[~mc] = bc_non.astype(str)
            except Exception:
                bb = xb.astype(object).where(~mb, "MISSING")
                bc = xc.astype(object).where(~mc, "MISSING")
        pb = bb.value_counts(normalize=True); pc = bc.value_counts(normalize=True)
        cats = list(pb.index.union(pc.index))
        psi = 0.0
        for k in cats:
            p = max(float(pb.get(k, 0.0)), EPS)
            q = max(float(pc.get(k, 0.0)), EPS)
            psi += (p - q) * float(np.log(p / q))
        return float(psi)

    fpd7_num = y.reindex(_feat2.index)

    iv_gt = iv_overall_s[iv_overall_s > 0.1].sort_values(ascending=False)
    rows = []
    for feat in iv_gt.index.tolist():
        x = pd.to_numeric(X[feat], errors="coerce")
        miss = x.isna() | (x == SENTINEL_VALUE)
        valid = (~miss) & fpd7_num.notna()
        corr = float(pd.Series(x[valid]).corr(fpd7_num[valid], method="pearson")) if int(valid.sum()) >= 3 else float("nan")
        rows.append(
            {
                "feature_name": feat,
                "iv": float(iv_gt.loc[feat]),
                "corr_fpd7": corr,
                "na_rate_na_or_-999": float(miss.mean()),
                "psi_last2w_vs_firstw": _psi_one(x) if (first_w is not None and len(last_two) > 0) else float("nan"),
                "iv_cycle_pass": float(iv_cycle_s.get(feat, float("nan"))),
                "iv_single_pass": float(iv_single_s.get(feat, float("nan"))),
            }
        )
    iv_report = pd.DataFrame(rows)
    for c in ["iv", "corr_fpd7", "na_rate_na_or_-999", "psi_last2w_vs_firstw", "iv_cycle_pass", "iv_single_pass"]:
        if c in iv_report.columns:
            iv_report[c] = pd.to_numeric(iv_report[c], errors="coerce").round(4)
    TABLES["iv_gt_0p1_report"] = iv_report

    top_overall = str(iv_overall_s.sort_values(ascending=False).index[0]) if len(iv_overall_s) > 0 else ""
    top_cycle = str(iv_cycle_s.sort_values(ascending=False).index[0]) if len(iv_cycle_s) > 0 else ""
    top_single = str(iv_single_s.sort_values(ascending=False).index[0]) if len(iv_single_s) > 0 else ""

    def _bin_report(feat: str, mask: pd.Series, method: str) -> pd.DataFrame:
        x = pd.to_numeric(X[feat], errors="coerce")[mask]
        yb = y01.reindex(X.index)[mask]
        miss = x.isna() | (x == SENTINEL_VALUE)
        x_non = x[~miss]
        if x_non.shape[0] == 0:
            lab = pd.Series("MISSING", index=x.index, dtype="object")
        else:
            if method == "qcut":
                r = x_non.rank(method="first")
                b_non = pd.qcut(r, q=BIN_N, duplicates="drop")
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
            else:
                b_non = pd.cut(x_non, bins=BIN_N, include_lowest=True)
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
        box = pd.DataFrame({"bin": lab, "y": yb}).dropna(subset=["y"])
        out = box.groupby("bin", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        uniq = pd.DataFrame({"bin": lab, "x": x.where(~miss, np.nan)}).groupby("bin", observed=False)["x"].nunique(dropna=True).rename("unique_cnt").reset_index()
        out = out.merge(uniq, on="bin", how="left")
        out["unique_cnt"] = out["unique_cnt"].fillna(0).astype(int)
        out.insert(0, "feature", feat)
        out.insert(1, "method", method)
        return out

    def _add_bins(tag: str, feat: str, mask: pd.Series):
        if not feat:
            return
        TABLES[f"bin_{tag}_qcut"] = _bin_report(feat, mask, "qcut")
        TABLES[f"bin_{tag}_cut"] = _bin_report(feat, mask, "cut")

    _add_bins("overall", top_overall, mask_all)
    _add_bins("cycle_pass", top_cycle, mask_cycle)
    _add_bins("single_pass", top_single, mask_single)

    def _weekly_iv(feat: str, mask: pd.Series) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "x": pd.to_numeric(X[feat], errors="coerce").reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out_rows = []
        for wk, g in tmp.groupby("week", observed=False):
            out_rows.append({"week": wk, "iv": _compute_iv_one(g["x"], g["y"]), "total_cnt": int(g.shape[0]), "bad_cnt": int(g["y"].sum())})
        out = pd.DataFrame(out_rows).sort_values("week")
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["iv"] = pd.to_numeric(out["iv"], errors="coerce").round(4)
        return out

    for tag, feat in [("overall_top", top_overall), ("cycle_top", top_cycle), ("single_top", top_single)]:
        if not feat:
            continue
        TABLES[f"weekly_iv_{tag}_overall"] = _weekly_iv(feat, mask_all)
        TABLES[f"weekly_iv_{tag}_cycle_pass"] = _weekly_iv(feat, mask_cycle)
        TABLES[f"weekly_iv_{tag}_single_pass"] = _weekly_iv(feat, mask_single)

# 4) 元信息
meta = pd.DataFrame(
    {
        "item": ["feature_cnt", "has_fpd7", "has_approve_state", "na_rate_ge_0.9_cnt", "same_value_rate_ge_0.99_cnt"],
        "value": [
            str(int(X.shape[1])),
            str(bool(has_fpd7)),
            str(bool(has_state)),
            str(int((na_rate >= 0.9).sum())),
            str(int((same_value_rate >= 0.99).sum())),
        ],
    }
)

# ========== 写 Excel（多 sheet） ==========
if WRITE_REPORTS:
    REPORT_XLSX.parent.mkdir(parents=True, exist_ok=True)
    used: set[str] = set()
    try:
        writer = pd.ExcelWriter(REPORT_XLSX, engine="openpyxl")
    except Exception:
        writer = pd.ExcelWriter(REPORT_XLSX)

    with writer:
        meta.to_excel(writer, sheet_name=_safe_sheet_name("meta", used), index=False)
        for name, df in TABLES.items():
            if df is None:
                df = pd.DataFrame()
            df.to_excel(writer, sheet_name=_safe_sheet_name(name, used), index=False)

    print("[WRITE_EXCEL] boss eval report saved:", str(REPORT_XLSX.resolve()))

# =========================
# 额外：生成 quality 检测 Excel（每个特征一行）
# =========================
# 大白话：按你的要求，BOSS notebook 也能独立输出 quality 表。
# 注意：BOSS 的特征字典 boss_feature_dict_3col 的 feature_name 是带 cdc_*_607 前后缀的；
#       quality 表里我们仍然按“去前后缀后的 base 名”做 feature_name，便于你与其它板块拼接。

WRITE_QUALITY_EXCEL = True
QUALITY_XLSX = Path("outputs") / "blockboss_feature_quality.xlsx"

if WRITE_QUALITY_EXCEL:
    import scripts.build_all_blocks_feature_quality_excel as q

    _base = raw_df[["apply_id", "request_time", "fpd7", "approve_state"]].copy()
    _base["apply_id"] = _base["apply_id"].astype(str)
    _base = _base.drop_duplicates("apply_id").set_index("apply_id")

    _y = (pd.to_numeric(_base["fpd7"], errors="coerce") > 0).astype(int)
    _state = _base["approve_state"].fillna("").astype(str).str.lower()
    _is_cycle = _state.str.contains("cycle_pass", na=False)
    _is_single = _state.str.contains("single_pass", na=False)

    _rt = pd.to_datetime(_base["request_time"], errors="coerce")
    _week_start = (_rt - pd.to_timedelta(_rt.dt.weekday, unit="D")).dt.normalize()
    _weeks = sorted([w for w in _week_start.dropna().unique()])
    _first_w = _weeks[0] if len(_weeks) >= 1 else None
    _last_two = _weeks[-2:] if len(_weeks) >= 2 else _weeks
    _base_mask = (_week_start == _first_w) if _first_w is not None else pd.Series(False, index=_week_start.index)
    _comp_mask = _week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=_week_start.index)

    # features_df 里有 apply_id 列，这里统一设 index。
    _feat = features_df.copy()
    if "apply_id" in _feat.columns:
        _feat = _feat.set_index("apply_id")
    _feat.index = _feat.index.astype(str)
    _X = _feat.drop(columns=["request_time"], errors="ignore")

    # BOSS 字典既支持 full key（cdc_*_607）也支持 base key（去前后缀）。
    _d = boss_feature_dict_3col.copy()
    _d["feature_name"] = _d["feature_name"].astype(str)
    _d["feature_name_base"] = _d["feature_name"].map(q._strip_feature_name)
    _d = _d.drop_duplicates("feature_name")
    _d_full = _d.set_index("feature_name")[["cn_desc", "logic_desc"]]
    _d_base = _d.drop_duplicates("feature_name_base").set_index("feature_name_base")[["cn_desc", "logic_desc"]]

    _rows = []
    _total = int(_X.shape[0])

    for _col in _X.columns:
        _col_name = str(_col)
        _base_name = q._strip_feature_name(_col_name)  # cdc_*_607 -> *

        _x = pd.to_numeric(_X[_col], errors="coerce")
        _miss = _x.isna() | (_x == q.SENTINEL)
        _na_cnt = int(_miss.sum())
        _na_rate = float(_na_cnt / _total) if _total else float("nan")
        _non = _x[~_miss]
        _nunique = int(_non.nunique(dropna=True))

        if _col_name in _d_full.index:
            _cn = str(_d_full.loc[_col_name, "cn_desc"])
            _logic = str(_d_full.loc[_col_name, "logic_desc"])
        elif _base_name in _d_base.index:
            _cn = str(_d_base.loc[_base_name, "cn_desc"])
            _logic = str(_d_base.loc[_base_name, "logic_desc"])
        else:
            _cn = ""
            _logic = ""

        _corr = float("nan")
        _valid = (~_miss) & _y.notna()
        if int(_valid.sum()) >= 3:
            _corr = q._corr_pearson(_x[_valid].to_numpy(dtype="float64", copy=False), _y[_valid].to_numpy(dtype="float64", copy=False))

        _iv = q._iv_one(_x, _y)
        _psi = q._psi_one(_x, _base_mask, _comp_mask)
        _iv_c = q._iv_one(_x[_is_cycle], _y[_is_cycle])
        _iv_s = q._iv_one(_x[_is_single], _y[_is_single])

        _rows.append(
            {
                "feature_name": _base_name,
                "cn_desc": _cn,
                "logic_desc": _logic,
                "na_cnt": _na_cnt,
                "total_cnt": _total,
                "na_rate": round(_na_rate, 6),
                "nunique": _nunique,
                "corr_y": None if pd.isna(_corr) else round(float(_corr), 6),
                "iv": None if pd.isna(_iv) else round(float(_iv), 6),
                "psi_last2w_vs_firstw": None if pd.isna(_psi) else round(float(_psi), 6),
                "iv_cycle_pass": None if pd.isna(_iv_c) else round(float(_iv_c), 6),
                "iv_single_pass": None if pd.isna(_iv_s) else round(float(_iv_s), 6),
                "notebook": "BOSS板块衍生.ipynb",
            }
        )

    _quality = pd.DataFrame(_rows)
    QUALITY_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(QUALITY_XLSX) as _w:
        _quality.to_excel(_w, sheet_name="report", index=False)
    print("[WRITE_EXCEL] boss feature quality saved:", str(QUALITY_XLSX.resolve()))



[WRITE_EXCEL] boss feature quality saved: /Users/zhanglifeng12703/Documents/OverseasPython/CDC/outputs/blockboss_feature_quality.xlsx
